# Phase 6 · Abstractive Summarization

Task: legal document → abstractive summary. Metric: BERTScore F1 (secondary: ROUGE-1, ROUGE-L). Output: [{"id", "summary"}].
Rules (§12): start from the Phase 1 models and use all 4. At least 6 substantially different experiments: at least 5 fine-tuning techniques (LoRMA compulsory) plus 1 KD (teacher → smaller student). No external pretrained models, corpora or APIs. Long documents must be handled explicitly (chunking / hierarchical / selection).
P6-0 downloads the data and profiles it. It checks whether the repo contains official scoring scripts; measures document and summary lengths (words and our tokens) and the compression ratio; measures how abstractive the reference summaries are (share of summary words and bigrams not in the document); finds which parts of the document the summaries draw from; computes a lead-k baseline; checks for leakage between train and test; and builds a fixed 90/10 train/validation split plus a one-time token cache.

In [1]:
# ============================================================================================
# P6-0: Phase 6 download + profile + leakage check + train/val split + token cache (CPU, ~5-10 min)
#   Writes data_p6/clean/{train,val,test}.json ({id[, summary]}) and data_p6/tok/{split}_{doc,sum}.npz
# ============================================================================================
import os, sys, re, json, glob, hashlib, random, time, shutil
import numpy as np
from collections import Counter
from multiprocessing import Pool
from huggingface_hub import snapshot_download, HfApi
from kaggle_secrets import UserSecretsClient

WORK, TOKEN = "/kaggle/working", UserSecretsClient().get_secret("HF_WRITE_TOKEN")
REPO = "Exploration-Lab/CS-781-Capstone"
sys.path.insert(0, f"{WORK}/src")

# ---- 1. Phase-1 assets present? (restore with R1 if not) -----------------------------------------
need = [f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json", f"{WORK}/src/common.py"] + \
       [f"{WORK}/checkpoints/{m}/best.pt" for m in ["m1_dense_100m", "m2_mla_moe", "m3_encdec", "m4_copy"]]
print("assets OK:", all(map(os.path.exists, need)), "| missing:", [os.path.relpath(p, WORK) for p in need if not os.path.exists(p)])

# ---- 2. what does the dataset repo contain for Phase 6 (and any official scoring scripts)? --------
files = HfApi(token=TOKEN).list_repo_files(REPO, repo_type="dataset")
print("Phase 6 files:", [f for f in files if f.startswith("Phase_6")])
print("scoring / eval files in the repo:", [f for f in files if re.search(r"(eval|scor|metric|bertscore)", f, re.I)] or "none")

# ---- 3. download + load (accepts list / {id: {...}} / jsonl) -------------------------------------
snapshot_download(REPO, repo_type="dataset", token=TOKEN, allow_patterns=["Phase_6_Summ/**"], local_dir=f"{WORK}/data_p6")
def load(path):
    if path.endswith(".jsonl"):
        return [json.loads(l) for l in open(path)]
    obj = json.load(open(path))
    return [{"id": k, **v} for k, v in obj.items()] if isinstance(obj, dict) else obj
S = {}
for f in sorted(glob.glob(f"{WORK}/data_p6/**/*.json*", recursive=True)):
    S[os.path.basename(f).split(".")[0]] = load(f)
    print(f"   {os.path.relpath(f, WORK)}  {os.path.getsize(f)/1e6:.1f} MB")
for n, recs in S.items():
    ex = {k: (v[:120] + " ...") if isinstance(v, str) and len(v) > 120 else v for k, v in recs[0].items()}
    print(f"=== {n}: {len(recs)} records | keys {list(recs[0])}\n    example: {ex}")
DOC = lambda r: r.get("document") or r.get("text") or r.get("judgment") or ""
SUM = lambda r: r.get("summary")
TR, TE = S["train"], S["test"]
print("test has summaries:", any(SUM(r) for r in TE), "(must be False)")

# ---- 4. sizes in words (what the summaries look like) ------------------------------------------
W = lambda t: len(t.split())
dw, sw, tw = np.array([W(DOC(r)) for r in TR]), np.array([W(SUM(r)) for r in TR]), np.array([W(DOC(r)) for r in TE])
q = lambda a: f"median {int(np.median(a))} | p10 {int(np.percentile(a, 10))} | p90 {int(np.percentile(a, 90))} | max {a.max()}"
print(f"\ntrain documents (words): {q(dw)}\ntrain summaries (words): {q(sw)}\ntest documents  (words): {q(tw)}")
print(f"compression (summary / document words): median {np.median(sw / np.maximum(dw, 1)):.3f}")
print(f"corr(document length, summary length): {np.corrcoef(dw, sw)[0, 1]:.2f}  (high -> generation length should scale with input)")

# ---- 5. how abstractive are the references? where does their content come from? ----------------
tok = lambda t: re.findall(r"[a-z0-9]+", t.lower())
rng = random.Random(0); sample = rng.sample(range(len(TR)), min(400, len(TR)))
nov1, nov2, dec = [], [], np.zeros(10)
for i in sample:
    d, s = tok(DOC(TR[i])), tok(SUM(TR[i]))
    if len(d) < 20 or len(s) < 5:
        continue
    d1, s1 = set(d), s; nov1.append(np.mean([w not in d1 for w in s1]))
    d2 = {(a, b): j for j, (a, b) in enumerate(zip(d, d[1:]))}                  # last position of each bigram
    s2 = list(zip(s, s[1:])); nov2.append(np.mean([b not in d2 for b in s2]))
    for b in set(s2):                                                           # which part of the document
        if b in d2:                                                             # the summary's bigrams match
            dec[min(9, int(10 * d2[b] / len(d)))] += 1
print(f"\nsummary words NOT in the document: {np.mean(nov1):.1%} | summary bigrams NOT in the document: {np.mean(nov2):.1%}")
print("share of matched summary bigrams by document decile (start -> end):", np.round(dec / dec.sum(), 3).tolist())

# ---- 6. simple reference points on a sample: lead-k words (k = that doc's expected summary length) --
def rouge(ref, hyp, cap=600):
    r, h = tok(ref)[:cap], tok(hyp)[:cap]
    if not r or not h:
        return 0.0, 0.0, 0.0
    def f(o, a, b): return 0.0 if o == 0 else 2 * o / (a + b)
    c1 = sum((Counter(r) & Counter(h)).values())
    r2, h2 = Counter(zip(r, r[1:])), Counter(zip(h, h[1:])); c2 = sum((r2 & h2).values())
    prev = [0] * (len(h) + 1)                                                    # LCS for ROUGE-L
    for x in r:
        cur = [0]
        for j, y in enumerate(h):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return f(c1, len(r), len(h)), f(c2, len(r) - 1, len(h) - 1), f(prev[-1], len(r), len(h))
ratio = float(np.median(sw / np.maximum(dw, 1)))
R = np.array([rouge(SUM(TR[i]), " ".join(DOC(TR[i]).split()[:max(30, int(ratio * W(DOC(TR[i]))))])) for i in sample[:200]])
print(f"lead-k baseline (approx. ROUGE, no stemming): R1 {R[:,0].mean():.3f} | R2 {R[:,1].mean():.3f} | RL {R[:,2].mean():.3f}")

# ---- 7. leakage: identical documents across train/test, duplicates inside train -------------------
h = lambda t: hashlib.md5(" ".join(t.split()).encode()).hexdigest()
Htr, Hte = [h(DOC(r)) for r in TR], {h(DOC(r)) for r in TE}
print(f"\ntrain docs identical to a test doc: {sum(x in Hte for x in Htr)} | duplicate docs inside train: {len(Htr) - len(set(Htr))}")
print("ids unique in test:", len({r['id'] for r in TE}) == len(TE))

# ---- 8. clean train (drop test-identical + duplicate docs) and a fixed 90/10 train/val split -----
seen, clean = set(Hte), []
for r, x in zip(TR, Htr):
    if x not in seen and DOC(r).strip() and SUM(r) and SUM(r).strip():
        seen.add(x); clean.append(r)
idx = list(range(len(clean))); random.Random(781).shuffle(idx); nv = max(50, len(clean) // 10)
SPL = {"val": [clean[i] for i in idx[:nv]], "train": [clean[i] for i in idx[nv:]], "test": TE}
print(f"clean: train {len(SPL['train'])} | val {len(SPL['val'])} (held out from train) | test {len(TE)}")

# ---- 9. tokenise once with our A1 BPE (documents and summaries separately, 4 processes) ----------
def _init():
    global _tk, _norm
    from tokenizers import Tokenizer
    from common import normalise as _n
    _tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json"); _norm = _n
def _enc(t):
    return np.asarray(_tk.encode(_norm(t), add_special_tokens=False).ids, dtype=np.uint16)
os.makedirs(f"{WORK}/data_p6/tok", exist_ok=True); os.makedirs(f"{WORK}/data_p6/clean", exist_ok=True)
for name, recs in SPL.items():
    t0 = time.time()
    parts = [("doc", [DOC(r) for r in recs])] + ([("sum", [SUM(r) for r in recs])] if name != "test" else [])
    for kind, texts in parts:
        with Pool(4, initializer=_init) as pool:
            arrs = pool.map(_enc, texts, chunksize=16)
        lens = np.array([len(a) for a in arrs], dtype=np.int64)
        np.savez(f"{WORK}/data_p6/tok/{name}_{kind}.npz", ids=np.concatenate(arrs), offsets=np.concatenate([[0], np.cumsum(lens)]))
        print(f"   {name} {kind}: {len(recs)} | tokens median {int(np.median(lens))} p90 {int(np.percentile(lens, 90))} "
              f"max {lens.max()} | <=1024: {np.mean(lens <= 1024):.0%} <=2048: {np.mean(lens <= 2048):.0%} | {time.time()-t0:.0f}s")
    json.dump([{"id": r["id"], **({"summary": SUM(r)} if name != "test" else {})} for r in recs],
              open(f"{WORK}/data_p6/clean/{name}.json", "w"))                    # test summaries never exist
print(f"\nfree disk: {shutil.disk_usage(WORK).free/1e9:.1f} GB")

assets OK: True | missing: []
Phase 6 files: ['Phase_6_Summ/Test/test.json', 'Phase_6_Summ/Train/train.json']
scoring / eval files in the repo: none


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

   data_p6/Phase_6_Summ/Test/test.json  2.7 MB
   data_p6/Phase_6_Summ/Train/train.json  210.6 MB
=== test: 100 records | keys ['id', 'document']
    example: {'id': '4963', 'document': 'Tax Reference Cases Nos. 3 to 5 of 1975 Tax Reference under Section 257 of the Income tax Act, 1961 made by the Income t ...'}
=== train: 7030 records | keys ['id', 'document', 'summary']
    example: {'id': '6712', 'document': 'IVIL Appeal No.768 (NT) of 1977 etc. From the judgment Order dated 9.10.1975 of the Madhya Pradesh High Court in M.C.C.  ...', 'summary': 'The appellant, a manufacturer of cement, entered into an agreement with the Cement Manufacturing Company of India Limite ...'}
test has summaries: False (must be False)

train documents (words): median 3128 | p10 1348 | p90 7851 | max 139645
train summaries (words): median 638 | p10 293 | p90 1501 | max 27768
test documents  (words): median 3171 | p10 1503 | p90 8562 | max 39695
compression (summary / document words): median 0.217
corr(docum

## P6-C · Free disk (Phase 3–5 files are on HF)

Deletes Phase 3/4/5 files only if they’re already on your HF repo, plus the organizers’ raw Phase 3–5 data, which can be downloaded again. Keeps the four Phase 1 best.pt models, the tokenizer, the code, stats/ and Phase 6 data.

In [3]:
# ---- P6-C: free disk -- delete Phase 3/4/5 files that are safely on HF (or re-downloadable raw data) ----
import os, glob, shutil
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
WORK, REPO = "/kaggle/working", "Utkarsh-1718-1/cs781-phase1-work"
on_hf = set(HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN")).list_repo_files(REPO))
KEEP = {f"checkpoints/{m}/best.pt" for m in ("m1_dense_100m", "m2_mla_moe", "m3_encdec", "m4_copy")}
DIRS = ["p3_ckpt", "p3_lp", "p3_emb", "p4_ckpt", "p4_prob", "p4_chunks", "p5_ckpt", "p5_prob", "p5_chunks",
        "data_p3", "data_p4", "data_p5", "submissions_p2", "submissions_p3", "submissions_p4", "submissions_p5", "checkpoints"]
freed, kept = 0, []
for d in DIRS:
    for f in glob.glob(f"{WORK}/{d}/**/*", recursive=True):
        if not os.path.isfile(f):
            continue
        rel = os.path.relpath(f, WORK)
        if rel in KEEP:
            continue                                                  # the 4 Phase-1 models stay
        raw_org = "/Phase_" in "/" + rel                              # organisers' raw data: re-downloadable
        if rel in on_hf or raw_org:
            freed += os.path.getsize(f); os.remove(f)
        else:
            kept.append(rel)
print(f"freed {freed/1e9:.2f} GB | free now: {shutil.disk_usage(WORK).free/1e9:.1f} GB")
print(f"kept {len(kept)} files that are NOT on HF (left untouched):", kept[:15])

freed 10.17 GB | free now: 11.2 GB
kept 9 files that are NOT on HF (left untouched): ['p5_chunks/m1_partial4_first1024_td.pkl', 'p5_chunks/m1_full_first1024_td.pkl', 'checkpoints/abl_untied/run_config.json', 'checkpoints/abl_untied/latest.pt', 'checkpoints/abl_noqk/run_config.json', 'checkpoints/abl_noqk/latest.pt', 'checkpoints/m4_copy/run_config.json', 'checkpoints/abl_base/run_config.json', 'checkpoints/abl_base/latest.pt']


## P6-1 · Window-level training pairs

Documents are split into consecutive 520-word windows. Each reference-summary sentence is assigned to the window with the largest word-bigram overlap (unigram overlap breaks ties), and each window’s target is its sentences, in summary order. Inputs are capped at 640 tokens and targets at 384, so input + target ≤ 1,024 tokens. Test documents are windowed the same way.

In [4]:
# ============================================================================================
# P6-1: chunk-level training pairs (CPU, ~5-10 min)
#   Each document is cut into consecutive windows of WIN words. Every summary sentence is assigned to
#   the window it overlaps most (word bigrams, then unigrams). A window's target = its sentences, in
#   summary order. Test time: summarise every window, concatenate -> length scales with the document.
#   Writes data_p6/chunks/{split}.npz: in_ids/in_off (window tokens), tg_ids/tg_off (target tokens),
#   doc (document index), pos (window index, n_win) ; test has no targets.
# ============================================================================================
import os, sys, re, json, time, random
import numpy as np
from collections import Counter
from multiprocessing import Pool
WORK = "/kaggle/working"; sys.path.insert(0, f"{WORK}/src")
WIN, IN_MAX, TG_MAX = 520, 640, 384          # words per window | max input / target tokens (640+384 = 1024)

raw = {r["id"]: r for f in ("Train/train.json", "Test/test.json")
       for r in json.load(open(f"{WORK}/data_p6/Phase_6_Summ/{f}"))}
SPL = {s: [m["id"] for m in json.load(open(f"{WORK}/data_p6/clean/{s}.json"))] for s in ("train", "val", "test")}
SENT = re.compile(r"(?<=[.;:?!])\s+(?=[A-Z(\"'])")                    # rough sentence splitter
words = lambda t: re.findall(r"[a-z0-9]+", t.lower())

def windows(doc):
    w = doc.split()
    return [" ".join(w[i:i + WIN]) for i in range(0, max(1, len(w)), WIN)]

def align(wins, summary):
    """Assign each summary sentence to the window with the largest bigram (then unigram) overlap."""
    sents = [s for s in SENT.split(summary) if s.strip()]
    W1 = [set(words(x)) for x in wins]; W2 = [set(zip(words(x), words(x)[1:])) for x in wins]
    tg = [[] for _ in wins]
    for s in sents:
        u = words(s); b = set(zip(u, u[1:]))
        score = [(len(b & W2[k]), len(set(u) & W1[k])) for k in range(len(wins))]
        tg[max(range(len(wins)), key=lambda k: score[k])].append(s)
    return [" ".join(t) for t in tg]

def _init():
    global _tk, _norm
    from tokenizers import Tokenizer
    from common import normalise as _n
    _tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json"); _norm = _n
def _enc(t):
    return np.asarray(_tk.encode(_norm(t), add_special_tokens=False).ids, dtype=np.uint16) if t else np.zeros(0, np.uint16)
pack = lambda arrs: (np.concatenate(arrs) if arrs else np.zeros(0, np.uint16),
                     np.concatenate([[0], np.cumsum([len(a) for a in arrs])]).astype(np.int64))

os.makedirs(f"{WORK}/data_p6/chunks", exist_ok=True)
for split, ids in SPL.items():
    t0 = time.time(); ins, tgs, doc, pos = [], [], [], []
    for d, i in enumerate(ids):
        wins = windows(raw[i]["document"])
        tg = align(wins, raw[i]["summary"]) if split != "test" else [""] * len(wins)
        for k, (x, y) in enumerate(zip(wins, tg)):
            ins.append(x); tgs.append(y); doc.append(d); pos.append((k, len(wins)))
    with Pool(4, initializer=_init) as pool:
        I = [a[:IN_MAX] for a in pool.map(_enc, ins, chunksize=64)]
        T = pool.map(_enc, tgs, chunksize=64) if split != "test" else []
    tl = np.array([len(a) for a in T]) if T else np.zeros(0)
    T = [a[:TG_MAX] for a in T]
    np.savez(f"{WORK}/data_p6/chunks/{split}.npz", **dict(zip(("in_ids", "in_off"), pack(I))),
             **(dict(zip(("tg_ids", "tg_off"), pack(T))) if T else {}), doc=np.array(doc), pos=np.array(pos))
    msg = f"{split:5s}: {len(ids)} docs -> {len(I)} windows ({len(I)/len(ids):.1f}/doc) | input tokens median {int(np.median([len(a) for a in I]))}"
    if len(tl):
        msg += (f" | target tokens median {int(np.median(tl))} p90 {int(np.percentile(tl, 90))} | empty targets {np.mean(tl == 0):.0%}"
                f" | truncated at {TG_MAX}: {np.mean(tl > TG_MAX):.1%}")
    print(msg + f" | {time.time()-t0:.0f}s")

# sanity: one validation document, its windows and aligned targets
z = np.load(f"{WORK}/data_p6/chunks/val.npz"); m = z["doc"] == 0
print("\nval doc 0:", int(m.sum()), "windows | target tokens per window:",
      [int(z["tg_off"][j + 1] - z["tg_off"][j]) for j in np.flatnonzero(m)])

train: 6301 docs -> 55897 windows (8.9/doc) | input tokens median 597 | target tokens median 82 p90 281 | empty targets 25% | truncated at 384: 3.3% | 84s
val  : 700 docs -> 6408 windows (9.2/doc) | input tokens median 598 | target tokens median 73 p90 275 | empty targets 27% | truncated at 384: 3.2% | 11s
test : 100 docs -> 968 windows (9.7/doc) | input tokens median 599 | 1s

val doc 0: 5 windows | target tokens per window: [234, 0, 316, 111, 100]


## P6-2 · Summarization module + runner

p6_sum.py: one code path for all four Phase 1 models.

Decoder-only (M1, M2, M4): [window] <sep> [target] <eos>, with the loss on the target only.
M3: encoder ← window, decoder ← target. M3 was pretrained with exactly this 640/384 split.
M4: the exact copy-head mixture loss (vocabulary vs pointing at a word in the window), restricted to the target.

Methods (plain PyTorch, no PEFT library): full, LoRA, LoRMA+ / LoRMA-π, Houlsby adapters (after attention and FFN), Pfeiffer adapters (after FFN only), prompt tuning, and QLoRA (our own 4-bit NF4 quantization of the frozen weights, two codes per byte, plus LoRA).

KD: a 6-layer M1 student learns from the full M1 teacher’s token distributions.

Generation: greedy KV-cached decoding with early stopping at <eos>, an n-gram repeat block, and an eos_bias for length control. Window outputs are joined in document order, and duplicate sentences are removed (redundancy reduction).

p6_run.py: background runner, same pattern as before. After each experiment it records validation ROUGE-1/2/L on 100 held-out documents, the length ratio, trainable parameters, model memory, generation speed and validation NLL, then writes the test zip.

In [5]:
!pip install -q rouge-score

In [6]:
%%writefile /kaggle/working/src/p6_sum.py
"""
p6_sum.py -- Phase 6 (abstractive legal summarisation) on ANY of the four Phase-1 models.

Strategy ("window-level summarisation"): every judgment is cut into ~520-word windows (<= 640 tokens);
each window is trained to produce ITS part of the reference summary (the summary sentences that
overlap it most, built in P6-1; empty for windows the summary skips). At test time every window is
summarised and the pieces are joined in document order -> output length scales with input length.

  decoder-only  M1 / M2 / M4 : [window] <sep> [target] <eos>         loss on target + <eos> only
  encoder-dec   M3           : encoder <- window, decoder <- <bos> target, predicts target <eos>
  M4 (copy head): exact mixture NLL (vocabulary vs pointer into the window), masked to the target

Adaptation methods (apply_method): full | lora | lorma_plus | lorma_pi | houlsby (adapter after the
attention AND the FFN sub-layer) | pfeiffer (adapter after the FFN only) | prompt (soft prompt,
M1/M2/M3) | qlora (frozen weights quantised to 4-bit NF4, double-packed in uint8, + LoRA).
Knowledge distillation: student (M1 with 6 of 12 layers) <- teacher logits (word-level KD).
Everything is plain PyTorch; no PEFT / transformers library.
"""
from __future__ import annotations

import contextlib
import math
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import p2_peft as pp

EOS, PAD, SEP, BOS = 0, 1, 3, 5
IN_MAX, TG_MAX = 640, 384
DEC_ONLY = ("dense", "mla_moe", "copy")


# ----------------------------------------------------------------------------- data
class Chunks:
    """Window-level pairs from P6-1 (data_p6/chunks/<split>.npz)."""
    def __init__(self, path):
        z = np.load(path)
        self.in_ids, self.in_off = z["in_ids"], z["in_off"]
        self.has_tg = "tg_ids" in z
        if self.has_tg:
            self.tg_ids, self.tg_off = z["tg_ids"], z["tg_off"]
        self.doc, self.pos = z["doc"], z["pos"]
        self.n = len(self.doc)

    def inp(self, i):
        return self.in_ids[self.in_off[i]:self.in_off[i + 1]].astype(np.int64)

    def tgt(self, i):
        return self.tg_ids[self.tg_off[i]:self.tg_off[i + 1]].astype(np.int64) if self.has_tg else np.zeros(0, np.int64)

    def windows_of(self, docs):
        s = set(docs)
        return [i for i in range(self.n) if self.doc[i] in s]


class Trunc(Chunks):
    """Long-document BASELINE: one pair per document = first 640 document tokens -> first 384
    summary tokens (data_p6/tok/<split>_{doc,sum}.npz)."""
    def __init__(self, doc_npz, sum_npz=None):
        d = np.load(doc_npz); n = len(d["offsets"]) - 1
        I = [d["ids"][d["offsets"][i]:d["offsets"][i + 1]][:IN_MAX] for i in range(n)]
        self.in_ids = np.concatenate(I); self.in_off = np.concatenate([[0], np.cumsum([len(a) for a in I])])
        self.has_tg = sum_npz is not None
        if self.has_tg:
            s = np.load(sum_npz)
            T = [s["ids"][s["offsets"][i]:s["offsets"][i + 1]][:TG_MAX] for i in range(n)]
            self.tg_ids = np.concatenate(T); self.tg_off = np.concatenate([[0], np.cumsum([len(a) for a in T])])
        self.doc, self.pos, self.n = np.arange(n), np.zeros((n, 2), dtype=np.int64), n


# ----------------------------------------------------------------------------- models
def load_lm(ckpt):
    from train import build_model
    ck = torch.load(ckpt, map_location="cpu", weights_only=False)
    m = build_model(ck["arch"], ck["model_cfg"])
    m.load_state_dict({k: v.float() for k, v in ck["model"].items()})
    m.arch = ck["arch"]; m.n_soft = 0
    return m


def layers(m):
    return list(m.enc) + list(m.dec) if m.arch == "encdec" else list(m.blocks)


def shrink(m, keep=(0, 2, 4, 6, 8, 10)):
    """KD student: keep every other layer of a decoder (DistilBERT-style initialisation)."""
    m.blocks = nn.ModuleList([m.blocks[i] for i in keep])
    return m


# ----------------------------------------------------------------------------- adaptation methods
LORA_TARGETS = ("wq", "wv", "w_uv", "wo")


def _wrap(m, make, targets):
    n = 0
    for L in layers(m):
        for mod in list(L.modules()):
            for name in targets:
                lin = getattr(mod, name, None)
                if lin is not None and hasattr(lin, "in_features") and not isinstance(lin, (pp.LoRALinear,)):
                    setattr(mod, name, make(lin)); n += 1
    return n


class Adapter(nn.Module):
    """Bottleneck adapter on a sub-layer's output: y + U relu(D y), U = 0 at start."""
    def __init__(self, sub, d, bottleneck=64):
        super().__init__()
        self.sub = sub
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, 0.0, 0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, *a, **kw):
        y = self.sub(*a, **kw)
        return y + self.up(F.relu(self.down(y)))


def add_adapters(m, kind, bottleneck=64):
    """houlsby: after attention AND FFN sub-layers (Houlsby et al. 2019); pfeiffer: after the FFN
    only (Pfeiffer et al. 2021)."""
    d, n = m.cfg.d_model, 0
    attn_names = ("attn", "self_attn") if kind == "houlsby" else ()
    for L in layers(m):
        for name in attn_names + ("mlp", "ffn"):
            sub = getattr(L, name, None)
            if sub is not None:
                setattr(L, name, Adapter(sub, d, bottleneck)); n += 1
    return n


class SoftPrompt(nn.Module):
    """Prompt tuning (Lester et al. 2021): n trainable vectors prepended to the INPUT side only.
    Active only while `on` is set (prefill / encoder), never for later decoding steps."""
    def __init__(self, emb, n=20):
        super().__init__()
        self.emb, self.n, self.on = emb, n, False
        idx = torch.randint(1000, 30000, (n,))
        self.prompt = nn.Parameter(emb.weight[idx].detach().clone())

    @property
    def weight(self):                                                  # keeps lm_head tying intact
        return self.emb.weight

    def forward(self, ids):
        x = self.emb(ids)
        if not self.on:
            return x
        return torch.cat([self.prompt.unsqueeze(0).expand(x.size(0), -1, -1).to(x.dtype), x], 1)


@contextlib.contextmanager
def soft_on(m):
    sp = m.tok_emb if isinstance(m.tok_emb, SoftPrompt) else None
    if sp is not None:
        sp.on = True
    try:
        yield
    finally:
        if sp is not None:
            sp.on = False


NF4 = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
                    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
                    0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
                    0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """QLoRA base layer (Dettmers et al. 2023): frozen weight stored as 4-bit NormalFloat codes,
    two codes per uint8, one absmax scale per block of 64 weights; de-quantised on the fly."""
    def __init__(self, lin: nn.Linear, block=64):
        super().__init__()
        W = lin.weight.detach().float()
        self.out_features, self.in_features, self.block = W.shape[0], W.shape[1], block
        flat = W.reshape(-1); pad = (-flat.numel()) % block
        flat = torch.cat([flat, flat.new_zeros(pad)]).view(-1, block)
        scale = flat.abs().amax(1, keepdim=True).clamp(min=1e-8)
        q = flat / scale
        codes = torch.cat([(q[i:i + 4096, :, None] - NF4.to(q.device)).abs().argmin(-1)
                           for i in range(0, q.size(0), 4096)]).to(torch.uint8).view(-1)
        if codes.numel() % 2:
            codes = torch.cat([codes, codes.new_zeros(1)])
        self.register_buffer("packed", (codes[0::2] << 4) | codes[1::2])
        self.register_buffer("scale", scale.half())
        self.register_buffer("lut", NF4.clone())
        self.n = W.numel()

    def weight_fp(self, dtype):
        c = torch.stack([self.packed >> 4, self.packed & 15], 1).view(-1).long()
        w = (self.lut[c].view(-1, self.block) * self.scale.float()).view(-1)[:self.n]
        return w.view(self.out_features, self.in_features).to(dtype)

    def forward(self, x):
        return F.linear(x, self.weight_fp(x.dtype))


def quantize_nf4(m):
    n = 0
    for L in layers(m):
        for mod in list(L.modules()):
            for name, child in list(mod.named_children()):
                if isinstance(child, nn.Linear) and name != "router":
                    setattr(mod, name, NF4Linear(child)); n += 1
    return n


def apply_method(m, method, r=16, alpha=32, bottleneck=64, n_prompt=20):
    """Freeze the Phase-1 model and add the method's trainable parameters."""
    if method == "full":
        for p in m.parameters():
            p.requires_grad = True
        return m
    for p in m.parameters():
        p.requires_grad = False
    if method == "lora":
        _wrap(m, lambda l: pp.LoRALinear(l, r, alpha), LORA_TARGETS)
    elif method == "lorma_plus":
        _wrap(m, lambda l: pp.LoRMAPlusLinear(l, r, alpha), LORA_TARGETS)
    elif method == "lorma_pi":
        _wrap(m, lambda l: pp.LoRMAPiLinear(l, r), LORA_TARGETS)
    elif method in ("houlsby", "pfeiffer"):
        add_adapters(m, method, bottleneck)
    elif method == "prompt":
        assert m.arch != "copy", "prompt tuning is not wired into M4's pointer cache"
        m.tok_emb = SoftPrompt(m.tok_emb, n_prompt); m.n_soft = n_prompt
        if m.arch == "encdec":                                         # M3: soft prompt on the ENCODER only
            enc = m.encode

            def encode_with_prompt(src):
                with soft_on(m):
                    return enc(src)
            m.encode = encode_with_prompt
    elif method == "qlora":
        quantize_nf4(m)
        _wrap(m, lambda l: pp.LoRALinear(l, r, alpha), LORA_TARGETS)
    else:
        raise ValueError(method)
    if m.arch == "copy":                                               # M4's small copy head always learns
        for mod in (m.copy_q, m.copy_k, m.gate):
            for p in mod.parameters():
                p.requires_grad = True
    return m


def n_params(m):
    seen = {id(p): p for p in m.parameters()}
    tot = sum(p.numel() for p in seen.values())
    tr = sum(p.numel() for p in seen.values() if p.requires_grad)
    mem = sum(p.numel() * p.element_size() for p in seen.values()) + \
        sum(b.numel() * b.element_size() for n, b in m.named_buffers() if "rope" not in n)
    return tr, tot, mem


# ----------------------------------------------------------------------------- batching + loss
def make_batch(arch, pairs, device):
    """pairs: [(input ids, target ids)]."""
    if arch == "encdec":
        L = min(len(x) for x, _ in pairs)                              # no encoder padding: trim to the
        src = torch.tensor(np.stack([x[:L] for x, _ in pairs]))       # shortest window of the batch
        Td = max(len(y) for _, y in pairs) + 1
        din = torch.full((len(pairs), Td), PAD, dtype=torch.long); y = torch.full((len(pairs), Td), -1, dtype=torch.long)
        for b, (_, t) in enumerate(pairs):
            seq = list(t) + [EOS]
            din[b, :len(seq)] = torch.tensor([BOS] + list(t)); y[b, :len(seq)] = torch.tensor(seq)
        return {"src": src.to(device), "din": din.to(device), "y": y.to(device)}
    seqs = [np.concatenate([x, [SEP], t, [EOS]]) for x, t in pairs]
    T = max(len(s) for s in seqs) - 1
    x = torch.full((len(pairs), T), PAD, dtype=torch.long)
    y = torch.full((len(pairs), T), -1, dtype=torch.long)              # loss targets (-1 = ignored)
    yf = torch.full((len(pairs), T), -2, dtype=torch.long)             # next token everywhere (copy keys)
    for b, (s, (inp, _)) in enumerate(zip(seqs, pairs)):
        s = torch.tensor(s); n = len(s) - 1
        x[b, :n] = s[:-1]; yf[b, :n] = s[1:]
        y[b, len(inp):n] = s[1:][len(inp):]                            # from <sep> onwards: target + <eos>
    return {"x": x.to(device), "y": y.to(device), "yf": yf.to(device)}


def _copy_nll(m, h, yf, y):
    """M4 mixture NLL with a loss mask (Phase-1 copy_loss assumes every position is a target)."""
    B, T, _ = h.shape
    z = m.lm_head(h)
    lp_vocab = -F.cross_entropy(z.float().view(-1, z.size(-1)), yf.clamp(min=0).view(-1), reduction="none").view(B, T)
    with torch.autocast("cuda", enabled=False):
        hf = h.float()
        allowed = torch.ones(T, T, dtype=torch.bool, device=h.device).tril(-1)
        g = m.gate(hf).squeeze(-1).masked_fill(~allowed.any(-1), -1e4)
        s = m.copy_q(hf) @ m.copy_k(hf).transpose(1, 2) / math.sqrt(m.cfg.copy_dim)
        la = torch.log_softmax(s.masked_fill(~allowed, -1e4), -1)
        match = (yf[:, None, :] == yf[:, :, None]) & (yf[:, None, :] >= 0)
        lp_copy = torch.logsumexp(la.masked_fill(~(match & allowed), -1e4), -1)
        lp = torch.logaddexp(F.logsigmoid(-g) + lp_vocab, F.logsigmoid(g) + lp_copy)
    mask = y >= 0
    return -(lp * mask).sum() / mask.sum().clamp(min=1)


def logits_and_targets(m, batch):
    """(logits at loss positions (N, V), targets (N,)) for CE / KD; M4 handled by loss()."""
    if m.arch == "encdec":
        mem = m.encode(batch["src"])                                   # (soft prompt, if any, added inside)
        z = m.decode_tf(mem, batch["din"])
        y = batch["y"]
    else:
        with soft_on(m):
            z = m(batch["x"])
        z = z[:, m.n_soft:]
        y = batch["y"]
    mask = y >= 0
    return z[mask], y[mask]


def loss(m, batch):
    if m.arch == "copy":
        return _copy_nll(m, m._hidden(batch["x"]), batch["yf"], batch["y"])
    z, y = logits_and_targets(m, batch)
    return F.cross_entropy(z.float(), y)


# ----------------------------------------------------------------------------- training
def batches(data, idx, bs, seed, drop_empty=0.5):
    """Length-bucketed batches; drop a share of the empty-target windows (they are 25% of all)."""
    rng = random.Random(seed)
    keep = [i for i in idx if data.tg_off[i + 1] > data.tg_off[i] or rng.random() >= drop_empty]
    rng.shuffle(keep)
    out = []
    for c in range(0, len(keep), bs * 50):                             # sort inside blocks of 50 batches
        blk = sorted(keep[c:c + bs * 50], key=lambda i: data.in_off[i + 1] - data.in_off[i] +
                     data.tg_off[i + 1] - data.tg_off[i])
        out += [blk[b:b + bs] for b in range(0, len(blk), bs)]
    rng.shuffle(out)
    return out


def train(m, data, idx, epochs=1, lr=2e-4, bs=8, device="cuda", seed=0, warmup=0.03, teacher=None,
          kd_T=2.0, kd_lam=0.5, max_steps=None, log=print):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    m.to(device); params = [p for p in m.parameters() if p.requires_grad]
    if teacher is not None:
        teacher.to(device).eval()
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    plan = [b for ep in range(epochs) for b in batches(data, idx, bs, seed + ep)]
    if max_steps:
        plan = plan[:max_steps]
    steps = len(plan); warm = max(1, int(warmup * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=str(device).startswith("cuda"))
    tr, tot, _ = n_params(m)
    log(f"trainable {tr/1e6:.3f}M of {tot/1e6:.1f}M | {steps} steps" + (" | KD" if teacher is not None else ""))
    t0, run, hist = time.time(), 0.0, []
    for step, b in enumerate(plan):
        m.train()
        batch = make_batch(m.arch, [(data.inp(i), data.tgt(i)) for i in b], device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=str(device).startswith("cuda")):
            if teacher is None:
                L = loss(m, batch)
            else:
                zs, y = logits_and_targets(m, batch)
                with torch.no_grad():
                    zt, _ = logits_and_targets(teacher, batch)
                ce = F.cross_entropy(zs.float(), y)
                kl = F.kl_div(F.log_softmax(zs.float() / kd_T, -1), F.log_softmax(zt.float() / kd_T, -1),
                              log_target=True, reduction="batchmean") * kd_T * kd_T
                L = (1 - kd_lam) * ce + kd_lam * kl
        opt.zero_grad(set_to_none=True)
        scaler.scale(L).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); sched.step(); run += L.item()
        if hasattr(m, "after_step"):
            m.after_step()                                             # M2: aux-loss-free expert balancing
        if (step + 1) % 250 == 0:
            hist.append({"step": step + 1, "loss": run / 250})
            log(f"   step {step+1}/{steps} | loss {run/250:.4f} | {time.time()-t0:.0f}s"); run = 0.0
    return hist


@torch.no_grad()
def val_loss(m, data, idx, bs=8, device="cuda"):
    """Mean target NLL on held-out windows (teacher forcing) -> a cheap, length-free quality signal."""
    m.eval(); tot, n = 0.0, 0
    for c in range(0, len(idx), bs):
        b = idx[c:c + bs]
        batch = make_batch(m.arch, [(data.inp(i), data.tgt(i)) for i in b], device)
        with torch.autocast("cuda", dtype=torch.float16):
            tot += loss(m, batch).item() * len(b); n += len(b)
    return tot / max(1, n)


# ----------------------------------------------------------------------------- generation
@torch.no_grad()
def generate(m, data, idx, max_new=TG_MAX, bs=16, eos_bias=0.0, no_repeat=4, device="cuda", log=None):
    """Greedy KV-cached decoding of every window in idx; stops a batch when all rows emitted <eos>.
    eos_bias > 0 -> shorter outputs, < 0 -> longer. no_repeat: block repeated n-grams.
    Returns {window index: generated token list}."""
    m.eval(); out = {}
    order = sorted(idx, key=lambda i: data.in_off[i + 1] - data.in_off[i])
    for c in range(0, len(order), bs):
        b = order[c:c + bs]; B = len(b)
        L = min(int(data.in_off[i + 1] - data.in_off[i]) for i in b)
        src = torch.tensor(np.stack([data.inp(i)[:L] for i in b]), device=device)
        prompt = src if m.arch == "encdec" else torch.cat([src, torch.full((B, 1), SEP, device=device)], 1)
        T = prompt.size(1); ns = 0 if m.arch == "encdec" else m.n_soft
        with torch.autocast("cuda", dtype=torch.float16):
            caches = m.make_caches(B, T + ns + max_new + 2, device, dtype=torch.float16)
            with (soft_on(m) if m.arch != "encdec" else contextlib.nullcontext()):   # M3: encode() adds it
                z = m(prompt, caches=caches, pos=0)[:, -1].float()
            gen = torch.zeros(B, 0, dtype=torch.long, device=device)
            done = torch.zeros(B, dtype=torch.bool, device=device)
            for t in range(max_new):
                z[:, 1:16] = float("-inf")                                 # specials other than <eos>
                z[:, EOS] += eos_bias
                if no_repeat and gen.size(1) >= no_repeat - 1:
                    for r in range(B):
                        s = gen[r].tolist(); pre = tuple(s[-(no_repeat - 1):])
                        bad = [s[j + no_repeat - 1] for j in range(len(s) - no_repeat + 1) if tuple(s[j:j + no_repeat - 1]) == pre]
                        if bad:
                            z[r, bad] = float("-inf")
                nxt = z.argmax(-1)
                nxt = torch.where(done, torch.full_like(nxt, EOS), nxt)
                gen = torch.cat([gen, nxt[:, None]], 1); done |= nxt == EOS
                if bool(done.all()):
                    break
                z = m(nxt[:, None], caches=caches, pos=T + ns + t)[:, -1].float()
        for r, i in enumerate(b):
            s = gen[r].tolist()
            out[i] = s[:s.index(EOS)] if EOS in s else s
    return out


def join_summaries(data, gen, docs, decode, dedupe=True):
    """Per document: the windows' outputs in document order, joined; optionally drop sentences
    already produced by an earlier window (redundancy reduction)."""
    import re
    res = {}
    for d in docs:
        wins = sorted([i for i in gen if data.doc[i] == d], key=lambda i: data.pos[i][0])
        sents, seen = [], set()
        for i in wins:
            for s in re.split(r"(?<=[.;:?!])\s+", decode(gen[i]).strip()):
                k = " ".join(s.lower().split())
                if s and (not dedupe or k not in seen):
                    sents.append(s); seen.add(k)
        res[d] = " ".join(sents).strip() or "."
    return res


def rouge_scores(refs, hyps):
    """ROUGE-1/2/L F1 (rouge-score package, stemming) + generated/reference length ratio."""
    from rouge_score import rouge_scorer
    sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    R = [sc.score(r, h) for r, h in zip(refs, hyps)]
    out = {k: float(np.mean([x[k].fmeasure for x in R])) for k in ("rouge1", "rouge2", "rougeL")}
    out["len_ratio"] = float(np.mean([len(h.split()) / max(1, len(r.split())) for r, h in zip(refs, hyps)]))
    return out

Writing /kaggle/working/src/p6_sum.py


In [7]:
%%writefile /kaggle/working/src/p6_run.py
"""
p6_run.py -- background runner for Phase-6 summarisation experiments (one process per GPU):

    CUDA_VISIBLE_DEVICES=0 PYTHONUNBUFFERED=1 setsid nohup python p6_run.py --plan plan.json >> log 2>&1 &
    python p6_run.py --smoke        # ~3-5 min: every model x method x code path on a few windows

Job (default type "run"):
  {"name", "ckpt": "m1".."m4", "method": full|lora|lorma_plus|lorma_pi|houlsby|pfeiffer|prompt|qlora,
   "lr", "epochs", "bs", "r", "alpha", "mode": "chunk" (window-level, default) | "trunc" (baseline),
   KD: "teacher": <finished run name>, "student_layers": [0,2,4,6,8,10], "kd_T", "kd_lam"}
  {"type": "gen", "name": X, "tag": T, "eos_bias": b, "no_repeat": n, "dedupe": bool}
        re-decode finished run X with other generation settings -> X__T (length / redundancy study)
Outputs per run: stats/p6_runs/<n>.json (written LAST = done marker), p6_ckpt/<n>.pt (trained params),
p6_gen/<n>.json ({"val": {id: summary}, "test": {id: summary}}), submissions_p6/p6_<n>/p6_<n>.zip.
Validation = the first 100 documents of our held-out split (never trained on).
"""
import argparse
import json
import os
import sys
import time
import traceback
import zipfile

import numpy as np
import torch

WORK = "/kaggle/working"
sys.path.insert(0, f"{WORK}/src")
import p6_sum as S                                                    # noqa: E402
from tokenizers import Tokenizer                                     # noqa: E402

ap = argparse.ArgumentParser(); ap.add_argument("--plan"); ap.add_argument("--smoke", action="store_true")
ARGS = ap.parse_args()
DEV = "cuda:0"
CK = {"m1": f"{WORK}/checkpoints/m1_dense_100m/best.pt", "m2": f"{WORK}/checkpoints/m2_mla_moe/best.pt",
      "m3": f"{WORK}/checkpoints/m3_encdec/best.pt", "m4": f"{WORK}/checkpoints/m4_copy/best.pt"}
OUT, CKD, GEN, SUB = (f"{WORK}/stats/p6_runs", f"{WORK}/p6_ckpt", f"{WORK}/p6_gen", f"{WORK}/submissions_p6")
for d in (OUT, CKD, GEN, SUB):
    os.makedirs(d, exist_ok=True)
N_VAL = 100


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


TK = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
decode = lambda ids: TK.decode(ids, skip_special_tokens=True)
IDS = {s: [m["id"] for m in json.load(open(f"{WORK}/data_p6/clean/{s}.json"))] for s in ("train", "val", "test")}
REF = {m["id"]: m["summary"] for m in json.load(open(f"{WORK}/data_p6/clean/val.json"))}
_DATA = {}


def data(mode, split):
    key = (mode, split)
    if key not in _DATA:
        _DATA[key] = (S.Chunks(f"{WORK}/data_p6/chunks/{split}.npz") if mode == "chunk" else
                      S.Trunc(f"{WORK}/data_p6/tok/{split}_doc.npz",
                              f"{WORK}/data_p6/tok/{split}_sum.npz" if split != "test" else None))
    return _DATA[key]


def build(ckpt, method, student_layers=None, r=16, alpha=32, **_):
    m = S.load_lm(CK[ckpt])
    if student_layers:
        S.shrink(m, tuple(student_layers))
    return S.apply_method(m, method, r=r, alpha=alpha)


def load_run(name):
    R = json.load(open(f"{OUT}/{name}.json")); c = R["cfg"]
    m = build(**c)
    sd = {k: v.float() for k, v in torch.load(f"{CKD}/{name}.pt", map_location="cpu").items()}
    m.load_state_dict(sd, strict=False)
    return m.to(DEV), c


def summarise(m, mode, split, docs, **gen_kw):
    d = data(mode, split); idx = d.windows_of(docs)
    t0 = time.time(); g = S.generate(m, d, idx, device=DEV, **{k: v for k, v in gen_kw.items() if k != "dedupe"})
    secs = time.time() - t0
    text = S.join_summaries(d, g, docs, decode, dedupe=gen_kw.get("dedupe", True))
    return {IDS[split][k]: v for k, v in text.items()}, secs, sum(len(v) for v in g.values())


def finish(name, cfg, m, mode, t0, extra, gen_kw):
    val, vs, vtok = summarise(m, mode, "val", list(range(N_VAL)), **gen_kw)
    test, ts, ttok = summarise(m, mode, "test", list(range(len(IDS["test"]))), **gen_kw)
    rs = S.rouge_scores([REF[i] for i in val], [val[i] for i in val])
    tr, tot, mem = S.n_params(m)
    json.dump({"val": val, "test": test}, open(f"{GEN}/{name}.json", "w"))
    folder = f"{SUB}/p6_{name}"; os.makedirs(folder, exist_ok=True)
    pred = [{"id": i, "summary": test[i]} for i in IDS["test"]]
    assert len(pred) == len(set(IDS["test"])) and all(p["summary"].strip() for p in pred)
    json.dump(pred, open(f"{folder}/prediction.json", "w"))
    with zipfile.ZipFile(f"{folder}/p6_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{folder}/prediction.json", arcname="prediction.json")
    out = {"cfg": cfg, "gen": gen_kw, "trainable_M": round(tr / 1e6, 3), "total_M": round(tot / 1e6, 1),
           "model_MB": round(mem / 1e6, 1), "val_rouge": rs, "gen_tok_per_s": round((vtok + ttok) / max(1e-6, vs + ts), 1),
           "test_gen_min": round(ts / 60, 1), "test_words_median": int(np.median([len(v.split()) for v in test.values()])),
           "min": round((time.time() - t0) / 60, 1), **extra}
    json.dump(out, open(f"{OUT}/{name}.json", "w"), indent=1)                 # written LAST = done marker
    log(f"DONE {name}: val R1 {rs['rouge1']:.4f} R2 {rs['rouge2']:.4f} RL {rs['rougeL']:.4f} | length x{rs['len_ratio']:.2f} "
        f"| {out['gen_tok_per_s']} tok/s | {out['min']} min")


def run(name, ckpt="m1", method="full", lr=2e-4, epochs=1, bs=8, r=16, alpha=32, mode="chunk", teacher=None,
        student_layers=None, kd_T=2.0, kd_lam=0.5, max_steps=None, seed=0):
    t0 = time.time()
    cfg = dict(ckpt=ckpt, method=method, lr=lr, epochs=epochs, bs=bs, r=r, alpha=alpha, mode=mode,
               teacher=teacher, student_layers=student_layers, kd_T=kd_T, kd_lam=kd_lam, seed=seed)
    m = build(ckpt, method, student_layers, r, alpha)
    T = load_run(teacher)[0] if teacher else None
    tr_d, va_d = data(mode, "train"), data(mode, "val")
    log(f"=== {name}: {ckpt} | {method} | {mode} | lr {lr}" + (f" | KD from {teacher}" if teacher else ""))
    hist = S.train(m, tr_d, list(range(tr_d.n)), epochs=epochs, lr=lr, bs=bs, device=DEV, seed=seed, teacher=T,
                   kd_T=kd_T, kd_lam=kd_lam, max_steps=max_steps, log=log)
    del T; torch.cuda.empty_cache()
    vl = S.val_loss(m, va_d, va_d.windows_of(list(range(N_VAL))), device=DEV)
    log(f"   val target NLL {vl:.4f} | training {round((time.time()-t0)/60, 1)} min")
    torch.save({n: p.detach().half().cpu() for n, p in m.named_parameters() if p.requires_grad}, f"{CKD}/{name}.pt")
    finish(name, cfg, m, mode, t0, {"hist": hist, "val_nll": vl, "train_min": round((time.time() - t0) / 60, 1)},
           {"eos_bias": 0.0, "no_repeat": 4, "dedupe": True})
    del m; torch.cuda.empty_cache()


def gen_job(name, tag, **gen_kw):
    t0 = time.time(); m, c = load_run(name)
    log(f"=== {name}__{tag}: re-decode with {gen_kw}")
    finish(f"{name}__{tag}", {**c, "base": name}, m, c["mode"], t0, {}, {"eos_bias": 0.0, "no_repeat": 4, "dedupe": True, **gen_kw})
    del m; torch.cuda.empty_cache()


def smoke():
    """Every model x method on a few windows: 2 train steps, val NLL, 3 windows decoded (6 tokens)."""
    tr, va = data("chunk", "train"), data("chunk", "val")
    few = [i for i in range(200) if tr.tg_off[i + 1] > tr.tg_off[i]][:4]
    combos = [("m1", "full"), ("m1", "lorma_plus"), ("m1", "lorma_pi"), ("m1", "qlora"), ("m1", "prompt"),
              ("m2", "houlsby"), ("m3", "full"), ("m3", "pfeiffer"), ("m3", "prompt"), ("m4", "lora")]
    for ck, meth in combos:
        m = build(ck, meth); h = S.train(m, tr, few, lr=1e-4, bs=2, device=DEV, max_steps=2, log=lambda *a: None)
        vl = S.val_loss(m, va, va.windows_of([0])[:2], device=DEV)
        g = S.generate(m, va, va.windows_of([0])[:3], max_new=6, device=DEV)
        print(f"OK {ck} {meth:10s} | trainable {S.n_params(m)[0]/1e6:.2f}M | loss {h and h[-1]['loss'] or 'n/a'} "
              f"| val NLL {vl:.3f} | sample: {decode(next(iter(g.values())))!r}")
        del m; torch.cuda.empty_cache()
    t = S.apply_method(S.load_lm(CK["m1"]), "full"); s = build("m1", "full", (0, 2, 4, 6, 8, 10))
    S.train(s, tr, few, bs=2, device=DEV, max_steps=2, teacher=t, log=lambda *a: None)
    print("OK KD student <- teacher | student layers", len(s.blocks))
    tc = data("trunc", "val"); m = build("m3", "full")
    print("OK trunc baseline: pairs", tc.n, "| NLL", round(S.val_loss(m, tc, [0, 1], device=DEV), 3))
    print("OK rouge:", S.rouge_scores(["the court held that the appeal fails"], ["the appeal fails the court held"]))
    print("SMOKE TEST PASSED")


if ARGS.smoke:
    smoke(); sys.exit(0)
for job in json.load(open(ARGS.plan)):
    job = dict(job); name = job.pop("name"); typ = job.pop("type", "run")
    target = f"{name}__{job['tag']}" if typ == "gen" else name
    if os.path.exists(f"{OUT}/{target}.json"):
        log(f"skip {target} (already done)"); continue
    try:
        gen_job(name, **job) if typ == "gen" else run(name, **job)
    except Exception:
        log(f"!!! {target} FAILED"); traceback.print_exc(); torch.cuda.empty_cache()
log("PLAN DONE")

Writing /kaggle/working/src/p6_run.py


## P6-3 · Smoke test (GPU, ~5 min)

Every model × method combination (including QLoRA quantization, prompt tuning on M1 and M3, Houlsby on M2’s MoE, M4’s copy loss), plus KD, the truncation baseline and ROUGE. Each runs 2 training steps, a validation NLL and 6 decoded tokens. It must end with SMOKE TEST PASSED.

In [8]:
# ---- P6-3: smoke test -- every model x method x code path on a few windows (GPU on) ----
!cd /kaggle/working/src && python p6_run.py --smoke 2>&1 | tail -n 25

  File "/kaggle/working/src/p6_sum.py", line 391, in val_loss
    tot += loss(m, batch).item() * len(b); n += len(b)
           ^^^^^^^^^^^^^^
  File "/kaggle/working/src/p6_sum.py", line 322, in loss
    z, y = logits_and_targets(m, batch)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/src/p6_sum.py", line 307, in logits_and_targets
    mem = m.encode(batch["src"])                                   # (soft prompt, if any, added inside)
          ^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/src/model_encdec.py", line 140, in encode
    x = self.tok_emb(src)
        ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/

## P6-4 · Experiments (launch + keep-alive + backup)
GPU	experiment	Phase 1 model	technique	rough time
0	m1_full	M1	full fine-tuning (KD teacher)	~1.3 h
0	kd_m1x6	M1 6-layer	KD from m1_full (word-level, T=2)	~1.1 h
0	m1_qlora	M1	QLoRA (NF4 4-bit + LoRA)	~1.4 h
0	m3_trunc	M3	full, truncated input (long-document baseline)	~0.4 h
1	m1_lorma	M1	LoRMA+ (compulsory)	~1.1 h
1	m4_lora	M4 (copy head)	LoRA	~1.3 h
1	m3_full	M3 enc–dec	full fine-tuning	~1.0 h
1	m2_houlsby	M2 MLA+MoE	Houlsby adapters	~1.5 h
1	m3_pfeiffer	M3	Pfeiffer adapters	~0.8 h

That’s 6 listed techniques (LoRA, QLoRA, LoRMA, Houlsby, Pfeiffer, KD), all 4 models and both long-document strategies. Each run includes generating validation and test summaries (~15 min).

In [9]:
# ---- P6-4: launch both queues, keep the session busy until they finish, then back up ----
import json, subprocess, time
from IPython.display import clear_output
WORK = "/kaggle/working"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
assert __import__("torch").cuda.device_count() == 2, "turn on the T4 x2 accelerator first"
plans = {
    0: [dict(name="m1_full", ckpt="m1", method="full", lr=1e-4),
        dict(name="kd_m1x6", ckpt="m1", method="full", lr=2e-4, teacher="m1_full", student_layers=[0, 2, 4, 6, 8, 10]),
        dict(name="m1_qlora", ckpt="m1", method="qlora", lr=5e-4),
        dict(name="m3_trunc", ckpt="m3", method="full", lr=1e-4, mode="trunc", epochs=2)],
    1: [dict(name="m1_lorma", ckpt="m1", method="lorma_plus", lr=5e-4),
        dict(name="m4_lora", ckpt="m4", method="lora", lr=5e-4),
        dict(name="m3_full", ckpt="m3", method="full", lr=1e-4),
        dict(name="m2_houlsby", ckpt="m2", method="houlsby", lr=5e-4),
        dict(name="m3_pfeiffer", ckpt="m3", method="pfeiffer", lr=5e-4)],
}
for g, plan in plans.items():
    if sh(f"pgrep -f '[p]6_run.py --plan .*p6_gpu{g}.json'"):
        print(f"GPU {g}: already running"); continue
    pf = f"{WORK}/stats/p6_gpu{g}.json"; json.dump(plan, open(pf, "w"), indent=1)
    subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES={g} PYTHONUNBUFFERED=1 setsid nohup python p6_run.py "
                   f"--plan {pf} >> {WORK}/stats/p6_gpu{g}.log 2>&1 < /dev/null &", shell=True)
    print(f"GPU {g}: launched {[j['name'] for j in plan]}")

time.sleep(60)
while sh("pgrep -f '[p]6_run.py --plan'"):                          # keep-alive + compact progress
    clear_output(wait=True)
    print(f"[{time.strftime('%H:%M', time.gmtime(time.time() + 19800))} IST]")
    for g in (0, 1):
        print(f"GPU {g}: {sh(f'grep -c DONE {WORK}/stats/p6_gpu{g}.log')} done | {sh(f'grep -c FAILED {WORK}/stats/p6_gpu{g}.log')} failed"
              f" | {sh(f'tail -n 1 {WORK}/stats/p6_gpu{g}.log')[:140]}")
    print(sh(f"grep -h 'DONE ' {WORK}/stats/p6_gpu0.log {WORK}/stats/p6_gpu1.log | grep -v PLAN"))
    time.sleep(180)

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN")).upload_folder(
    repo_id="Utkarsh-1718-1/cs781-phase1-work", repo_type="model", folder_path=WORK,
    commit_message="phase 6 experiments", ignore_patterns=["**/__pycache__/**"],
    allow_patterns=["src/**", "stats/**", "p6_ckpt/**", "p6_gen/**", "submissions_p6/**",
                    "data_p6/clean/**", "data_p6/chunks/**", "data_p6/tok/**"])
print("backup done ✅")

[16:04 IST]
GPU 0: 1 done | 0 failed | 10:30:00    val target NLL 1.4135 | training 43.8 min
GPU 1: 2 done | 0 failed | 10:32:58    step 250/6107 | loss 2.8271 | 73s
09:46:11 DONE m1_full: val R1 0.4222 R2 0.2348 RL 0.2455 | length x0.49 | 129.5 tok/s | 61.0 min
09:33:18 DONE m1_lorma: val R1 0.2775 R2 0.1432 RL 0.1735 | length x0.27 | 81.2 tok/s | 48.1 min
10:31:42 DONE m4_lora: val R1 0.3781 R2 0.1986 RL 0.2169 | length x0.44 | 90.1 tok/s | 58.4 min


KeyboardInterrupt: 

## P6-5 · Length calibration (eos_bias sweep per model)

For every experiment: decode 20 validation documents at eos_bias ∈ {−1.5, −3, −4.5, −6}, pick the best (R1+RL)/2, then produce the calibrated validation + test summaries and zip (<run>__cal). Each job waits for its base run. Running this cell also keeps the session alive and backs up at the end.

In [10]:
%%writefile /kaggle/working/src/p6_run.py
"""
p6_run.py -- background runner for Phase-6 summarisation experiments (one process per GPU):

    CUDA_VISIBLE_DEVICES=0 PYTHONUNBUFFERED=1 setsid nohup python p6_run.py --plan plan.json >> log 2>&1 &
    python p6_run.py --smoke        # ~3-5 min: every model x method x code path on a few windows

Job (default type "run"):
  {"name", "ckpt": "m1".."m4", "method": full|lora|lorma_plus|lorma_pi|houlsby|pfeiffer|prompt|qlora,
   "lr", "epochs", "bs", "r", "alpha", "mode": "chunk" (window-level, default) | "trunc" (baseline),
   KD: "teacher": <finished run name>, "student_layers": [0,2,4,6,8,10], "kd_T", "kd_lam"}
  {"type": "gen", "name": X, "tag": T, "eos_bias": b, "no_repeat": n, "dedupe": bool}
        re-decode finished run X with other generation settings -> X__T (length / redundancy study)
  {"type": "cal", "name": X, "biases": [...], "n_docs": 20}
        LENGTH CALIBRATION: waits until run X is finished, decodes the first n_docs validation documents
        with every eos_bias in `biases`, keeps the one with the best (ROUGE-1 + ROUGE-L) / 2, then writes
        the full validation + test output with it -> X__cal (+ its zip)
Outputs per run: stats/p6_runs/<n>.json (written LAST = done marker), p6_ckpt/<n>.pt (trained params),
p6_gen/<n>.json ({"val": {id: summary}, "test": {id: summary}}), submissions_p6/p6_<n>/p6_<n>.zip.
Validation = the first 100 documents of our held-out split (never trained on).
"""
import argparse
import json
import os
import sys
import time
import traceback
import zipfile

import numpy as np
import torch

WORK = "/kaggle/working"
sys.path.insert(0, f"{WORK}/src")
import p6_sum as S                                                    # noqa: E402
from tokenizers import Tokenizer                                     # noqa: E402

ap = argparse.ArgumentParser(); ap.add_argument("--plan"); ap.add_argument("--smoke", action="store_true")
ARGS = ap.parse_args()
DEV = "cuda:0"
CK = {"m1": f"{WORK}/checkpoints/m1_dense_100m/best.pt", "m2": f"{WORK}/checkpoints/m2_mla_moe/best.pt",
      "m3": f"{WORK}/checkpoints/m3_encdec/best.pt", "m4": f"{WORK}/checkpoints/m4_copy/best.pt"}
OUT, CKD, GEN, SUB = (f"{WORK}/stats/p6_runs", f"{WORK}/p6_ckpt", f"{WORK}/p6_gen", f"{WORK}/submissions_p6")
for d in (OUT, CKD, GEN, SUB):
    os.makedirs(d, exist_ok=True)
N_VAL = 100


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


TK = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
decode = lambda ids: TK.decode(ids, skip_special_tokens=True)
IDS = {s: [m["id"] for m in json.load(open(f"{WORK}/data_p6/clean/{s}.json"))] for s in ("train", "val", "test")}
REF = {m["id"]: m["summary"] for m in json.load(open(f"{WORK}/data_p6/clean/val.json"))}
_DATA = {}


def data(mode, split):
    key = (mode, split)
    if key not in _DATA:
        _DATA[key] = (S.Chunks(f"{WORK}/data_p6/chunks/{split}.npz") if mode == "chunk" else
                      S.Trunc(f"{WORK}/data_p6/tok/{split}_doc.npz",
                              f"{WORK}/data_p6/tok/{split}_sum.npz" if split != "test" else None))
    return _DATA[key]


def build(ckpt, method, student_layers=None, r=16, alpha=32, **_):
    m = S.load_lm(CK[ckpt])
    if student_layers:
        S.shrink(m, tuple(student_layers))
    return S.apply_method(m, method, r=r, alpha=alpha)


def load_run(name):
    R = json.load(open(f"{OUT}/{name}.json")); c = R["cfg"]
    m = build(**c)
    sd = {k: v.float() for k, v in torch.load(f"{CKD}/{name}.pt", map_location="cpu").items()}
    m.load_state_dict(sd, strict=False)
    return m.to(DEV), c


def summarise(m, mode, split, docs, **gen_kw):
    d = data(mode, split); idx = d.windows_of(docs)
    t0 = time.time(); g = S.generate(m, d, idx, device=DEV, **{k: v for k, v in gen_kw.items() if k != "dedupe"})
    secs = time.time() - t0
    text = S.join_summaries(d, g, docs, decode, dedupe=gen_kw.get("dedupe", True))
    return {IDS[split][k]: v for k, v in text.items()}, secs, sum(len(v) for v in g.values())


def finish(name, cfg, m, mode, t0, extra, gen_kw):
    val, vs, vtok = summarise(m, mode, "val", list(range(N_VAL)), **gen_kw)
    test, ts, ttok = summarise(m, mode, "test", list(range(len(IDS["test"]))), **gen_kw)
    rs = S.rouge_scores([REF[i] for i in val], [val[i] for i in val])
    tr, tot, mem = S.n_params(m)
    json.dump({"val": val, "test": test}, open(f"{GEN}/{name}.json", "w"))
    folder = f"{SUB}/p6_{name}"; os.makedirs(folder, exist_ok=True)
    pred = [{"id": i, "summary": test[i]} for i in IDS["test"]]
    assert len(pred) == len(set(IDS["test"])) and all(p["summary"].strip() for p in pred)
    json.dump(pred, open(f"{folder}/prediction.json", "w"))
    with zipfile.ZipFile(f"{folder}/p6_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{folder}/prediction.json", arcname="prediction.json")
    out = {"cfg": cfg, "gen": gen_kw, "trainable_M": round(tr / 1e6, 3), "total_M": round(tot / 1e6, 1),
           "model_MB": round(mem / 1e6, 1), "val_rouge": rs, "gen_tok_per_s": round((vtok + ttok) / max(1e-6, vs + ts), 1),
           "test_gen_min": round(ts / 60, 1), "test_words_median": int(np.median([len(v.split()) for v in test.values()])),
           "min": round((time.time() - t0) / 60, 1), **extra}
    json.dump(out, open(f"{OUT}/{name}.json", "w"), indent=1)                 # written LAST = done marker
    log(f"DONE {name}: val R1 {rs['rouge1']:.4f} R2 {rs['rouge2']:.4f} RL {rs['rougeL']:.4f} | length x{rs['len_ratio']:.2f} "
        f"| {out['gen_tok_per_s']} tok/s | {out['min']} min")


def run(name, ckpt="m1", method="full", lr=2e-4, epochs=1, bs=8, r=16, alpha=32, mode="chunk", teacher=None,
        student_layers=None, kd_T=2.0, kd_lam=0.5, max_steps=None, seed=0):
    t0 = time.time()
    cfg = dict(ckpt=ckpt, method=method, lr=lr, epochs=epochs, bs=bs, r=r, alpha=alpha, mode=mode,
               teacher=teacher, student_layers=student_layers, kd_T=kd_T, kd_lam=kd_lam, seed=seed)
    m = build(ckpt, method, student_layers, r, alpha)
    T = load_run(teacher)[0] if teacher else None
    tr_d, va_d = data(mode, "train"), data(mode, "val")
    log(f"=== {name}: {ckpt} | {method} | {mode} | lr {lr}" + (f" | KD from {teacher}" if teacher else ""))
    hist = S.train(m, tr_d, list(range(tr_d.n)), epochs=epochs, lr=lr, bs=bs, device=DEV, seed=seed, teacher=T,
                   kd_T=kd_T, kd_lam=kd_lam, max_steps=max_steps, log=log)
    del T; torch.cuda.empty_cache()
    vl = S.val_loss(m, va_d, va_d.windows_of(list(range(N_VAL))), device=DEV)
    log(f"   val target NLL {vl:.4f} | training {round((time.time()-t0)/60, 1)} min")
    torch.save({n: p.detach().half().cpu() for n, p in m.named_parameters() if p.requires_grad}, f"{CKD}/{name}.pt")
    finish(name, cfg, m, mode, t0, {"hist": hist, "val_nll": vl, "train_min": round((time.time() - t0) / 60, 1)},
           {"eos_bias": 0.0, "no_repeat": 4, "dedupe": True})
    del m; torch.cuda.empty_cache()


def gen_job(name, tag, **gen_kw):
    t0 = time.time(); m, c = load_run(name)
    log(f"=== {name}__{tag}: re-decode with {gen_kw}")
    finish(f"{name}__{tag}", {**c, "base": name}, m, c["mode"], t0, {}, {"eos_bias": 0.0, "no_repeat": 4, "dedupe": True, **gen_kw})
    del m; torch.cuda.empty_cache()


def cal_job(name, biases=(-1.0, -2.0, -3.0, -4.5), n_docs=20, wait_h=8.0):
    t_wait = time.time()
    while not os.path.exists(f"{OUT}/{name}.json"):                    # base run still training elsewhere
        if time.time() - t_wait > wait_h * 3600:
            raise TimeoutError(f"{name} did not finish")
        time.sleep(120)
    t0 = time.time(); m, c = load_run(name); res = {}
    log(f"=== {name}__cal: eos_bias sweep {list(biases)} on {n_docs} val documents")
    for b in biases:
        val, _, _ = summarise(m, c["mode"], "val", list(range(n_docs)), eos_bias=b, no_repeat=4, dedupe=True)
        rs = S.rouge_scores([REF[i] for i in val], [val[i] for i in val]); res[b] = rs
        log(f"   eos_bias {b:+.1f}: R1 {rs['rouge1']:.4f} RL {rs['rougeL']:.4f} | length x{rs['len_ratio']:.2f}")
    best = max(res, key=lambda b: res[b]["rouge1"] + res[b]["rougeL"])
    log(f"   -> eos_bias {best:+.1f}")
    finish(f"{name}__cal", {**c, "base": name}, m, c["mode"], t0, {"cal": {str(k): v for k, v in res.items()}},
           {"eos_bias": best, "no_repeat": 4, "dedupe": True})
    del m; torch.cuda.empty_cache()


def smoke():
    """Every model x method on a few windows: 2 train steps, val NLL, 3 windows decoded (6 tokens)."""
    tr, va = data("chunk", "train"), data("chunk", "val")
    few = [i for i in range(200) if tr.tg_off[i + 1] > tr.tg_off[i]][:4]
    combos = [("m1", "full"), ("m1", "lorma_plus"), ("m1", "lorma_pi"), ("m1", "qlora"), ("m1", "prompt"),
              ("m2", "houlsby"), ("m3", "full"), ("m3", "pfeiffer"), ("m3", "prompt"), ("m4", "lora")]
    for ck, meth in combos:
        m = build(ck, meth); h = S.train(m, tr, few, lr=1e-4, bs=2, device=DEV, max_steps=2, log=lambda *a: None)
        vl = S.val_loss(m, va, va.windows_of([0])[:2], device=DEV)
        g = S.generate(m, va, va.windows_of([0])[:3], max_new=6, device=DEV)
        print(f"OK {ck} {meth:10s} | trainable {S.n_params(m)[0]/1e6:.2f}M | loss {h and h[-1]['loss'] or 'n/a'} "
              f"| val NLL {vl:.3f} | sample: {decode(next(iter(g.values())))!r}")
        del m; torch.cuda.empty_cache()
    t = S.apply_method(S.load_lm(CK["m1"]), "full"); s = build("m1", "full", (0, 2, 4, 6, 8, 10))
    S.train(s, tr, few, bs=2, device=DEV, max_steps=2, teacher=t, log=lambda *a: None)
    print("OK KD student <- teacher | student layers", len(s.blocks))
    tc = data("trunc", "val"); m = build("m3", "full").to(DEV)
    print("OK trunc baseline: pairs", tc.n, "| NLL", round(S.val_loss(m, tc, [0, 1], device=DEV), 3))
    print("OK rouge:", S.rouge_scores(["the court held that the appeal fails"], ["the appeal fails the court held"]))
    print("SMOKE TEST PASSED")


if ARGS.smoke:
    smoke(); sys.exit(0)
for job in json.load(open(ARGS.plan)):
    job = dict(job); name = job.pop("name"); typ = job.pop("type", "run")
    target = f"{name}__{job['tag']}" if typ == "gen" else (f"{name}__cal" if typ == "cal" else name)
    if os.path.exists(f"{OUT}/{target}.json"):
        log(f"skip {target} (already done)"); continue
    try:
        if typ == "gen":
            gen_job(name, **job)
        elif typ == "cal":
            cal_job(name, **job)
        else:
            run(name, **job)
    except Exception:
        log(f"!!! {target} FAILED"); traceback.print_exc(); torch.cuda.empty_cache()
log("PLAN DONE")

Overwriting /kaggle/working/src/p6_run.py


In [11]:
# ---- P6-5: launch length calibration next to the training (2nd process per GPU), keep-alive, backup ----
import json, subprocess, time
from IPython.display import clear_output
WORK = "/kaggle/working"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
B = [-1.5, -3.0, -4.5, -6.0]                                            # eos_bias values to try
cal = {0: ["m1_full", "kd_m1x6", "m1_qlora", "m3_trunc"],              # same GPU as their training runs
       1: ["m1_lorma", "m4_lora", "m3_full", "m2_houlsby", "m3_pfeiffer"]}
for g, names in cal.items():
    if sh(f"pgrep -f '[p]6_run.py --plan .*p6cal_gpu{g}.json'"):
        print(f"GPU {g}: calibration already running"); continue
    pf = f"{WORK}/stats/p6cal_gpu{g}.json"
    json.dump([dict(type="cal", name=n, biases=B, n_docs=20) for n in names], open(pf, "w"), indent=1)
    subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES={g} PYTHONUNBUFFERED=1 setsid nohup python p6_run.py "
                   f"--plan {pf} >> {WORK}/stats/p6cal_gpu{g}.log 2>&1 < /dev/null &", shell=True)
    print(f"GPU {g}: calibration queued for {names} (each waits for its run)")

time.sleep(60)
while sh("pgrep -f '[p]6_run.py --plan'"):                          # training + calibration still running
    clear_output(wait=True)
    print(f"[{time.strftime('%H:%M', time.gmtime(time.time() + 19800))} IST]")
    for f in ("p6_gpu0", "p6_gpu1", "p6cal_gpu0", "p6cal_gpu1"):
        print(f"{f:11s}: {sh(f'grep -c FAILED {WORK}/stats/{f}.log')} failed | {sh(f'tail -n 1 {WORK}/stats/{f}.log')[:130]}")
    print(sh(f"grep -h -E 'DONE |-> eos_bias' {WORK}/stats/p6_gpu0.log {WORK}/stats/p6_gpu1.log "
             f"{WORK}/stats/p6cal_gpu0.log {WORK}/stats/p6cal_gpu1.log | grep -v PLAN"))
    time.sleep(180)

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN")).upload_folder(
    repo_id="Utkarsh-1718-1/cs781-phase1-work", repo_type="model", folder_path=WORK,
    commit_message="phase 6 experiments + length calibration", ignore_patterns=["**/__pycache__/**"],
    allow_patterns=["src/**", "stats/**", "p6_ckpt/**", "p6_gen/**", "submissions_p6/**",
                    "data_p6/clean/**", "data_p6/chunks/**", "data_p6/tok/**"])
print("backup done ✅")

[16:19 IST]
p6_gpu0    : 0 failed | 10:47:58    step 1500/6107 | loss 1.5134 | 740s
p6_gpu1    : 0 failed | 10:48:44    step 2750/6107 | loss 2.5551 | 1019s
p6cal_gpu0 : 0 failed | 10:48:28    eos_bias -3.0: R1 0.4340 RL 0.2394 | length x3.23
p6cal_gpu1 : 0 failed | 10:49:37    eos_bias -3.0: R1 0.4182 RL 0.2225 | length x3.03
09:46:11 DONE m1_full: val R1 0.4222 R2 0.2348 RL 0.2455 | length x0.49 | 129.5 tok/s | 61.0 min
10:35:32 DONE kd_m1x6: val R1 0.3472 R2 0.1830 RL 0.2051 | length x0.36 | 200.6 tok/s | 49.4 min
09:33:18 DONE m1_lorma: val R1 0.2775 R2 0.1432 RL 0.1735 | length x0.27 | 81.2 tok/s | 48.1 min
10:31:42 DONE m4_lora: val R1 0.3781 R2 0.1986 RL 0.2169 | length x0.44 | 90.1 tok/s | 58.4 min


KeyboardInterrupt: 

In [12]:
# ---- P6-5b: finer eos_bias grid + length-aware choice; restart ONLY the calibration processes ----
import json, subprocess, time
from IPython.display import clear_output
WORK = "/kaggle/working"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()

# 1) choose among settings with length x0.8-1.25 (BERTScore, the primary metric, punishes length mismatch)
f = f"{WORK}/src/p6_run.py"; s = open(f).read()
old = '    best = max(res, key=lambda b: res[b]["rouge1"] + res[b]["rougeL"])\n'
new = ('    ok = [b for b in res if 0.8 <= res[b]["len_ratio"] <= 1.25]       # length close to the reference\n'
       '    best = (max(ok, key=lambda b: res[b]["rouge1"] + res[b]["rougeL"]) if ok\n'
       '            else min(res, key=lambda b: abs(res[b]["len_ratio"] - 1)))\n')
if old in s:
    open(f, "w").write(s.replace(old, new)); print("selection rule patched")
else:
    print("already patched" if "len_ratio\"] <= 1.25" in s else "!! pattern not found - tell Claude")

# 2) stop the running calibration processes only (training queues keep going)
print("stopping:", sh("pgrep -af '[p]6_run.py --plan .*p6cal'") or "none")
sh("pkill -f '[p]6_run.py --plan .*p6cal'"); time.sleep(5)

# 3) relaunch with the finer grid (each job still waits for its training run)
B = [-0.5, -0.75, -1.0, -1.25]
cal = {0: ["m1_full", "kd_m1x6", "m1_qlora", "m3_trunc"],
       1: ["m1_lorma", "m4_lora", "m3_full", "m2_houlsby", "m3_pfeiffer"]}
for g, names in cal.items():
    pf = f"{WORK}/stats/p6cal_gpu{g}.json"
    json.dump([dict(type="cal", name=n, biases=B, n_docs=20) for n in names], open(pf, "w"), indent=1)
    subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES={g} PYTHONUNBUFFERED=1 setsid nohup python p6_run.py "
                   f"--plan {pf} >> {WORK}/stats/p6cal_gpu{g}.log 2>&1 < /dev/null &", shell=True)
    print(f"GPU {g}: calibration relaunched for {names} with eos_bias {B}")

# 4) keep-alive + progress, then backup (same as P6-5)
time.sleep(60)
while sh("pgrep -f '[p]6_run.py --plan'"):
    clear_output(wait=True)
    print(f"[{time.strftime('%H:%M', time.gmtime(time.time() + 19800))} IST]")
    for f_ in ("p6_gpu0", "p6_gpu1", "p6cal_gpu0", "p6cal_gpu1"):
        print(f"{f_:11s}: {sh(f'grep -c FAILED {WORK}/stats/{f_}.log')} failed | {sh(f'tail -n 1 {WORK}/stats/{f_}.log')[:130]}")
    print(sh(f"grep -h -E 'DONE |-> eos_bias' {WORK}/stats/p6_gpu0.log {WORK}/stats/p6_gpu1.log "
             f"{WORK}/stats/p6cal_gpu0.log {WORK}/stats/p6cal_gpu1.log | grep -v PLAN"))
    time.sleep(180)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN")).upload_folder(
    repo_id="Utkarsh-1718-1/cs781-phase1-work", repo_type="model", folder_path=WORK,
    commit_message="phase 6 experiments + length calibration", ignore_patterns=["**/__pycache__/**"],
    allow_patterns=["src/**", "stats/**", "p6_ckpt/**", "p6_gen/**", "submissions_p6/**",
                    "data_p6/clean/**", "data_p6/chunks/**", "data_p6/tok/**"])
print("backup done ✅")

[21:41 IST]
p6_gpu0    : 0 failed | 11:53:12 DONE m1_qlora: val R1 0.2741 R2 0.1395 RL 0.1712 | length x0.27 | 28.6 tok/s | 77.7 min
p6_gpu1    : 0 failed | 13:41:09 PLAN DONE
p6cal_gpu0 : 0 failed | 13:18:06 DONE m1_qlora__cal: val R1 0.5239 R2 0.2695 RL 0.2693 | length x1.10 | 84.7 tok/s | 57.3 min
p6cal_gpu1 : 0 failed | 15:07:48 PLAN DONE
09:46:11 DONE m1_full: val R1 0.4222 R2 0.2348 RL 0.2455 | length x0.49 | 129.5 tok/s | 61.0 min
10:35:32 DONE kd_m1x6: val R1 0.3472 R2 0.1830 RL 0.2051 | length x0.36 | 200.6 tok/s | 49.4 min
11:53:12 DONE m1_qlora: val R1 0.2741 R2 0.1395 RL 0.1712 | length x0.27 | 28.6 tok/s | 77.7 min
09:33:18 DONE m1_lorma: val R1 0.2775 R2 0.1432 RL 0.1735 | length x0.27 | 81.2 tok/s | 48.1 min
10:31:42 DONE m4_lora: val R1 0.3781 R2 0.1986 RL 0.2169 | length x0.44 | 90.1 tok/s | 58.4 min
11:29:09 DONE m3_full: val R1 0.3522 R2 0.1481 RL 0.1970 | length x0.43 | 80.6 tok/s | 57.5 min
13:09:26 DONE m2_houlsby: val R1 0.2860 R2 0.1536 RL 0.1785 | length x0.27 

KeyboardInterrupt: 

In [13]:
# ---- P6-D: is m3_trunc alive? (GPU-0 training queue) ----
import subprocess
WORK = "/kaggle/working"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
print("p6 processes:\n" + (sh("pgrep -af '[p]6_run.py'") or "NONE"))
print(sh("nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader"))
print("--- last 8 lines of GPU-0 training log ---\n" + sh(f"tail -n 8 {WORK}/stats/p6_gpu0.log"))
print("--- kernel OOM kills ---\n" + (sh("dmesg 2>/dev/null | grep -i -E 'killed process|out of memory' | tail -3") or "none visible"))

p6 processes:
1046 python3 p6_run.py --plan /kaggle/working/stats/p6cal_gpu0.json
0, 0 %, 153 MiB
1, 0 %, 0 MiB
--- last 8 lines of GPU-0 training log ---
11:15:07    step 4750/6107 | loss 1.4184 | 2369s
11:17:09    step 5000/6107 | loss 1.4311 | 2491s
11:19:03    step 5250/6107 | loss 1.4353 | 2606s
11:21:03    step 5500/6107 | loss 1.4394 | 2725s
11:23:04    step 5750/6107 | loss 1.4677 | 2846s
11:25:07    step 6000/6107 | loss 1.4332 | 2969s
11:26:31    val target NLL 1.3453 | training 51.0 min
11:53:12 DONE m1_qlora: val R1 0.2741 R2 0.1395 RL 0.1712 | length x0.27 | 28.6 tok/s | 77.7 min
--- kernel OOM kills ---
none visible


In [14]:
# ---- P6-6: re-run the truncation baseline + its calibration, keep-alive, backup ----
import json, subprocess, time
WORK = "/kaggle/working"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
sh("pkill -f '[p]6_run.py --plan .*p6cal_gpu0'")                         # the waiting calibration process
plan = [dict(name="m3_trunc", ckpt="m3", method="full", lr=1e-4, mode="trunc", epochs=2),
        dict(type="cal", name="m3_trunc", biases=[-0.5, -0.75, -1.0, -1.25], n_docs=20)]
pf = f"{WORK}/stats/p6_trunc.json"; json.dump(plan, open(pf, "w"), indent=1)
subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES=0 PYTHONUNBUFFERED=1 setsid nohup python p6_run.py "
               f"--plan {pf} >> {WORK}/stats/p6_trunc.log 2>&1 < /dev/null &", shell=True)
time.sleep(60)
while sh("pgrep -f '[p]6_run.py --plan'"):                          # keep-alive
    print(time.strftime("%H:%M"), "UTC |", sh(f"tail -n 1 {WORK}/stats/p6_trunc.log")[:140]); time.sleep(180)
print(sh(f"grep -h -E 'DONE|FAILED' {WORK}/stats/p6_trunc.log"))
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN")).upload_folder(
    repo_id="Utkarsh-1718-1/cs781-phase1-work", repo_type="model", folder_path=WORK,
    commit_message="phase 6 experiments", ignore_patterns=["**/__pycache__/**"],
    allow_patterns=["src/**", "stats/**", "p6_ckpt/**", "p6_gen/**", "submissions_p6/**",
                    "data_p6/clean/**", "data_p6/chunks/**", "data_p6/tok/**"])
print("backup done ✅")

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

backup done ✅


In [1]:
# ---- Check whether the calibration process (PID 1046) is still doing work ----
import subprocess, glob, os, time

# Process state, CPU use and elapsed time: R = running, S = sleeping/waiting
print(subprocess.run("ps -o pid,stat,%cpu,etime,cmd -p 1046", shell=True,
                     capture_output=True, text=True).stdout)

# Find the newest log files in stats/ and show when each was last written
logs = sorted(glob.glob("/kaggle/working/stats/*.log"), key=os.path.getmtime)[-3:]
for f in logs:
    age = (time.time() - os.path.getmtime(f)) / 60
    print(f"\n=== {os.path.basename(f)} | last written {age:.1f} min ago ===")
    # Last 6 lines of each recent log
    print(subprocess.run(f"tail -n 6 {f}", shell=True,
                         capture_output=True, text=True).stdout)

    PID STAT %CPU     ELAPSED CMD


=== p6_gpu0.log | last written 1.5 min ago ===
11:19:03    step 5250/6107 | loss 1.4353 | 2606s
11:21:03    step 5500/6107 | loss 1.4394 | 2725s
11:23:04    step 5750/6107 | loss 1.4677 | 2846s
11:25:07    step 6000/6107 | loss 1.4332 | 2969s
11:26:31    val target NLL 1.3453 | training 51.0 min
11:53:12 DONE m1_qlora: val R1 0.2741 R2 0.1395 RL 0.1712 | length x0.27 | 28.6 tok/s | 77.7 min


=== p4_gpu0.log | last written 1.5 min ago ===
16:24:34    epoch 5: loss 0.2393 | dev macro-F1 0.7947 @0.5 | 0.8058 @t=0.30
16:24:41    epoch 6: loss 0.1976 | dev macro-F1 0.7923 @0.5 | 0.7975 @t=0.33
16:24:49    epoch 7: loss 0.1570 | dev macro-F1 0.7807 @0.5 | 0.7958 @t=0.12
16:24:56    epoch 8: loss 0.1368 | dev macro-F1 0.7836 @0.5 | 0.7973 @t=0.12
16:24:56 DONE m1_full1024_hier: dev macro-F1 0.7947 @0.5 | 0.8058 @t=0.30 | attention by decile [0.0402, 0.0062, 0.0339, 0.1213, 0.018, 0.0949, 0.238, 0.1728, 0.2249, 0.0497] | 54.6 min
16:24:56 PLAN DONE


=== p5

In [2]:
# ---- Confirm every calibrated (__cal) result is saved before test generation ----
import glob, os, json

# All calibration-related files in stats/ (logs, plans, result JSONs)
for f in sorted(glob.glob("/kaggle/working/stats/*cal*")):
    print(f"{os.path.getsize(f)/1024:8.1f} KB  {os.path.basename(f)}")

# Last 12 lines of the calibration log(s): should list a DONE line for each __cal run
for f in sorted(glob.glob("/kaggle/working/stats/p6cal*.log")):
    print(f"\n=== {os.path.basename(f)} ===")
    print("".join(open(f).readlines()[-12:]))

     0.5 KB  p6cal_gpu0.json
     1.6 KB  p6cal_gpu0.log
     0.6 KB  p6cal_gpu1.json
     2.5 KB  p6cal_gpu1.log

=== p6cal_gpu0.log ===
12:04:25    eos_bias -0.8: R1 0.5040 RL 0.2635 | length x0.76
12:05:39    eos_bias -1.0: R1 0.5389 RL 0.2738 | length x0.95
12:06:56    eos_bias -1.2: R1 0.5677 RL 0.2912 | length x1.13
12:06:56    -> eos_bias -1.2
12:20:49 DONE kd_m1x6__cal: val R1 0.5548 R2 0.3057 RL 0.2884 | length x1.36 | 415.1 tok/s | 18.4 min
12:20:55 === m1_qlora__cal: eos_bias sweep [-0.5, -0.75, -1.0, -1.25] on 20 val documents
12:23:24    eos_bias -0.5: R1 0.4003 RL 0.2304 | length x0.49
12:25:59    eos_bias -0.8: R1 0.4413 RL 0.2449 | length x0.66
12:29:08    eos_bias -1.0: R1 0.4951 RL 0.2598 | length x0.82
12:32:52    eos_bias -1.2: R1 0.5219 RL 0.2650 | length x1.03
12:32:52    -> eos_bias -1.2
13:18:06 DONE m1_qlora__cal: val R1 0.5239 R2 0.2695 RL 0.2693 | length x1.10 | 84.7 tok/s | 57.3 min


=== p6cal_gpu1.log ===
14:07:23    eos_bias -1.0: R1 0.5252 RL 0.2780 | le

In [3]:
# ---- Show the calibration plan format so the test plan can reuse it ----
import json

# Print both plan files exactly as p6_run.py reads them
for f in ["/kaggle/working/stats/p6cal_gpu0.json", "/kaggle/working/stats/p6cal_gpu1.json"]:
    print(f"=== {f.split('/')[-1]} ===")
    print(json.dumps(json.load(open(f)), indent=2))

# Show which command-line options p6_run.py accepts (e.g. a test or predict mode)
import subprocess
print(subprocess.run("cd /kaggle/working/src && python3 p6_run.py --help", shell=True,
                     capture_output=True, text=True).stdout[-2500:])

=== p6cal_gpu0.json ===
[
  {
    "type": "cal",
    "name": "m1_full",
    "biases": [
      -0.5,
      -0.75,
      -1.0,
      -1.25
    ],
    "n_docs": 20
  },
  {
    "type": "cal",
    "name": "kd_m1x6",
    "biases": [
      -0.5,
      -0.75,
      -1.0,
      -1.25
    ],
    "n_docs": 20
  },
  {
    "type": "cal",
    "name": "m1_qlora",
    "biases": [
      -0.5,
      -0.75,
      -1.0,
      -1.25
    ],
    "n_docs": 20
  },
  {
    "type": "cal",
    "name": "m3_trunc",
    "biases": [
      -0.5,
      -0.75,
      -1.0,
      -1.25
    ],
    "n_docs": 20
  }
]
=== p6cal_gpu1.json ===
[
  {
    "type": "cal",
    "name": "m1_lorma",
    "biases": [
      -0.5,
      -0.75,
      -1.0,
      -1.25
    ],
    "n_docs": 20
  },
  {
    "type": "cal",
    "name": "m4_lora",
    "biases": [
      -0.5,
      -0.75,
      -1.0,
      -1.25
    ],
    "n_docs": 20
  },
  {
    "type": "cal",
    "name": "m3_full",
    "biases": [
      -0.5,
      -0.75,
      -1.0,
     

In [4]:
# ---- Find the job types p6_run.py supports and any test submissions already written ----
import subprocess, glob, os

# Every job "type" the runner dispatches on, with its line number (e.g. "cal", "train", "test")
print(subprocess.run(r"grep -n -E \"type\"\]|\[.type.\]|== *['\"][a-z_]+['\"]\" /kaggle/working/src/p6_run.py",
                     shell=True, capture_output=True, text=True).stdout)

# Lines that write test-set outputs, which shows where submissions are saved
print(subprocess.run(r"grep -n -i -E 'test|submission|csv' /kaggle/working/src/p6_run.py | head -40",
                     shell=True, capture_output=True, text=True).stdout)

# Any submission or prediction files already on disk, newest last
files = [f for f in glob.glob("/kaggle/working/**/*", recursive=True)
         if any(k in os.path.basename(f).lower() for k in ("sub", "pred", "test"))
         and f.endswith((".csv", ".json", ".jsonl", ".zip"))]
for f in sorted(files, key=os.path.getmtime):
    print(f"{os.path.getsize(f)/1024:9.1f} KB  {f.replace('/kaggle/working/', '')}")


16:        the full validation + test output with it -> X__cal (+ its zip)
18:p6_gen/<n>.json ({"val": {id: summary}, "test": {id: summary}}), submissions_p6/p6_<n>/p6_<n>.zip.
42:OUT, CKD, GEN, SUB = (f"{WORK}/stats/p6_runs", f"{WORK}/p6_ckpt", f"{WORK}/p6_gen", f"{WORK}/submissions_p6")
54:IDS = {s: [m["id"] for m in json.load(open(f"{WORK}/data_p6/clean/{s}.json"))] for s in ("train", "val", "test")}
64:                              f"{WORK}/data_p6/tok/{split}_sum.npz" if split != "test" else None))
93:    test, ts, ttok = summarise(m, mode, "test", list(range(len(IDS["test"]))), **gen_kw)
96:    json.dump({"val": val, "test": test}, open(f"{GEN}/{name}.json", "w"))
98:    pred = [{"id": i, "summary": test[i]} for i in IDS["test"]]
99:    assert len(pred) == len(set(IDS["test"])) and all(p["summary"].strip() for p in pred)
105:           "test_gen_min": round(ts / 60, 1), "test_words_median": int(np.median([len(v.split()) for v in test.values()])),
179:    print("SMOKE TEST PASSED

In [5]:
# ---- Verify the calibrated test predictions and build upload zips in priority order ----
import json, zipfile, os, numpy as np

WORK = "/kaggle/working"
# Test IDs in the order the runner uses (same source as IDS["test"] in p6_run.py)
test_ids = [m["id"] for m in json.load(open(f"{WORK}/data_p6/clean/test.json"))]

# Upload order: best validation ROUGE-1 first
ORDER = ["m1_full", "kd_m1x6", "m4_lora", "m2_houlsby", "m1_lorma", "m1_qlora", "m3_full", "m3_pfeiffer"]

os.makedirs(f"{WORK}/upload_p6", exist_ok=True)
print(f"{'#':>2} {'run':<20} {'n':>4} {'ok':>3} {'test words':>10} {'val words':>9}  zip")
for k, n in enumerate(ORDER, 1):
    run = f"{n}__cal"
    pred = json.load(open(f"{WORK}/submissions_p6/p6_{run}/prediction.json"))

    # Checks: same ID set as the test file, no duplicates, no empty summaries
    ids = [p["id"] for p in pred]
    ok = (sorted(ids) == sorted(test_ids) and len(ids) == len(set(ids))
          and all(p["summary"].strip() for p in pred))

    # Median summary length on test vs val (from p6_gen) — a large gap means test decoding drifted
    gen = json.load(open(f"{WORK}/p6_gen/{run}.json"))
    tw = int(np.median([len(p["summary"].split()) for p in pred]))
    vw = int(np.median([len(s.split()) for s in gen["val"].values()]))

    # Zip with prediction.json at the root of the archive, numbered by upload order
    zp = f"{WORK}/upload_p6/{k}_p6_{run}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{WORK}/submissions_p6/p6_{run}/prediction.json", "prediction.json")

    print(f"{k:>2} {run:<20} {len(pred):>4} {'Y' if ok else 'NO':>3} {tw:>10} {vw:>9}  {os.path.basename(zp)}")

 # run                     n  ok test words val words  zip
 1 m1_full__cal          100   Y        680       742  1_p6_m1_full__cal.zip
 2 kd_m1x6__cal          100   Y        838       838  2_p6_kd_m1x6__cal.zip
 3 m4_lora__cal          100   Y        704       783  3_p6_m4_lora__cal.zip
 4 m2_houlsby__cal       100   Y        874       781  4_p6_m2_houlsby__cal.zip
 5 m1_lorma__cal         100   Y        609       624  5_p6_m1_lorma__cal.zip
 6 m1_qlora__cal         100   Y        639       659  6_p6_m1_qlora__cal.zip
 7 m3_full__cal          100   Y        922       923  7_p6_m3_full__cal.zip
 8 m3_pfeiffer__cal      100   Y        548       670  8_p6_m3_pfeiffer__cal.zip


In [6]:
# ---- Re-decode kd_m1x6 and m2_houlsby at eos_bias -1.0 (current -1.25 outputs kept as backups) ----
import json, os, shutil, subprocess, glob

WORK = "/kaggle/working"
RUNS = {"kd_m1x6": 0, "m2_houlsby": 1}   # run -> GPU

for n in RUNS:
    run = f"{n}__cal"
    # Move every existing output of this run aside with a _b125 suffix (results, generations, submission)
    for src in glob.glob(f"{WORK}/stats/p6_runs/{run}*") + [f"{WORK}/p6_gen/{run}.json",
                                                             f"{WORK}/submissions_p6/p6_{run}"]:
        if os.path.exists(src):
            root, ext = os.path.splitext(src) if os.path.isfile(src) else (src, "")
            shutil.move(src, f"{root.replace(run, run + '_b125')}{ext}")
            print("backed up:", os.path.basename(src))

for n, gpu in RUNS.items():
    # Single-value bias grid -> the job picks -1.0 and decodes full val + test with it
    plan = f"{WORK}/stats/p6redo_gpu{gpu}.json"
    json.dump([{"type": "cal", "name": n, "biases": [-1.0], "n_docs": 20}], open(plan, "w"), indent=2)

    # Launch in the background on its own GPU; output goes to its own log
    cmd = (f"cd {WORK}/src && CUDA_VISIBLE_DEVICES={gpu} nohup python3 p6_run.py --plan {plan} "
           f"> {WORK}/stats/p6redo_gpu{gpu}.log 2>&1 &")
    subprocess.run(cmd, shell=True)
    print(f"launched {n} on GPU {gpu}")

backed up: kd_m1x6__cal.json
backed up: kd_m1x6__cal.json
backed up: p6_kd_m1x6__cal
backed up: m2_houlsby__cal.json
backed up: m2_houlsby__cal.json
backed up: p6_m2_houlsby__cal
launched kd_m1x6 on GPU 0
launched m2_houlsby on GPU 1


In [7]:
# ---- Check the re-decode jobs started, and show the Trunc loader code for the m3_trunc fix ----
import subprocess

# Last lines of each re-decode log: should show the eos_bias -1.0 sweep line, no traceback
print(subprocess.run("tail -n 4 /kaggle/working/stats/p6redo_gpu*.log", shell=True,
                     capture_output=True, text=True).stdout)

# GPU use: both should now be well above 0 %
print(subprocess.run("nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)

# Every place the truncated-input path loads data, with 25 lines around the first hit
print(subprocess.run("grep -n -i -E 'trunc|np.load|npz' /kaggle/working/src/p6_sum.py",
                     shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("grep -n -i -A 25 -m 1 'class Trunc' /kaggle/working/src/p6_sum.py",
                     shell=True, capture_output=True, text=True).stdout)

==> /kaggle/working/stats/p6redo_gpu0.log <==
10:13:06 === kd_m1x6__cal: eos_bias sweep [-1.0] on 20 val documents
10:14:06    eos_bias -1.0: R1 0.5389 RL 0.2738 | length x0.95
10:14:06    -> eos_bias -1.0

==> /kaggle/working/stats/p6redo_gpu1.log <==
10:13:08 === m2_houlsby__cal: eos_bias sweep [-1.0] on 20 val documents

0, 37 %, 5999 MiB
1, 34 %, 5293 MiB

40:    """Window-level pairs from P6-1 (data_p6/chunks/<split>.npz)."""
42:        z = np.load(path)
61:class Trunc(Chunks):
63:    summary tokens (data_p6/tok/<split>_{doc,sum}.npz)."""
64:    def __init__(self, doc_npz, sum_npz=None):
65:        d = np.load(doc_npz); n = len(d["offsets"]) - 1
68:        self.has_tg = sum_npz is not None
70:            s = np.load(sum_npz)

61:class Trunc(Chunks):
62-    """Long-document BASELINE: one pair per document = first 640 document tokens -> first 384
63-    summary tokens (data_p6/tok/<split>_{doc,sum}.npz)."""
64-    def __init__(self, doc_npz, sum_npz=None):
65-        d = np.load(doc

In [2]:
# ---- Fix the Trunc loader: read each npz array once instead of once per document ----
import re, time, numpy as np, subprocess

P = "/kaggle/working/src/p6_sum.py"
src = open(P).read()
shutil_backup = open(P + ".bak", "w").write(src)   # keep the original in case we need to roll back

# Document side: pull ids/offsets out of the npz once, then slice the in-memory arrays
old_doc = ('        d = np.load(doc_npz); n = len(d["offsets"]) - 1\n'
           '        I = [d["ids"][d["offsets"][i]:d["offsets"][i + 1]][:IN_MAX] for i in range(n)]')
new_doc = ('        d = np.load(doc_npz); ids, off = d["ids"], d["offsets"]; n = len(off) - 1\n'
           '        I = [ids[off[i]:off[i + 1]][:IN_MAX] for i in range(n)]')

# Summary side: same fix
old_sum = ('            s = np.load(sum_npz)\n'
           '            T = [s["ids"][s["offsets"][i]:s["offsets"][i + 1]][:TG_MAX] for i in range(n)]')
new_sum = ('            s = np.load(sum_npz); sids, soff = s["ids"], s["offsets"]\n'
           '            T = [sids[soff[i]:soff[i + 1]][:TG_MAX] for i in range(n)]')

assert old_doc in src and old_sum in src, "code differs from what was printed - don't patch blindly"
open(P, "w").write(src.replace(old_doc, new_doc).replace(old_sum, new_sum))
print("patched p6_sum.py (original saved as p6_sum.py.bak)")

# Quick check on CPU: building the train split should now take seconds, not minutes
import importlib, sys
sys.path.insert(0, "/kaggle/working/src"); import p6_sum; importlib.reload(p6_sum)
W = "/kaggle/working/data_p6/tok"
t = time.time(); ds = p6_sum.Trunc(f"{W}/train_doc.npz", f"{W}/train_sum.npz")
print(f"train Trunc: {ds.n} docs in {time.time()-t:.1f}s | input tokens {len(ds.in_ids):,} | target tokens {len(ds.tg_ids):,}")

# Does m3_trunc already have a checkpoint, and which plan defines its training job?
print(subprocess.run("ls -la /kaggle/working/p6_ckpt | grep -i trunc; "
                     "grep -l m3_trunc /kaggle/working/stats/p6*.json", shell=True,
                     capture_output=True, text=True).stdout or "no m3_trunc checkpoint or plan entry found")

AssertionError: code differs from what was printed - don't patch blindly

In [3]:
# ---- Is the Trunc fix already in p6_sum.py? ----
import subprocess
src = open("/kaggle/working/src/p6_sum.py").read()

print("fix already applied:", 'ids, off = d["ids"], d["offsets"]' in src)
print("old buggy line present:", 'd["ids"][d["offsets"][i]' in src)

# Show the current Trunc loader lines so we can see exactly what's there
print(subprocess.run("grep -n -A 12 'class Trunc' /kaggle/working/src/p6_sum.py",
                     shell=True, capture_output=True, text=True).stdout)

# Did the session actually reset? Logs from this afternoon would still be here if it didn't
print(subprocess.run("ls -la --time-style=+%H:%M /kaggle/working/stats/ | grep -E 'p6redo|p6new|p6_trunc_gpu0'",
                     shell=True, capture_output=True, text=True).stdout or "no afternoon logs -> session was reset")

fix already applied: True
old buggy line present: False
61:class Trunc(Chunks):
62-    """Long-document BASELINE: one pair per document = first 640 document tokens -> first 384
63-    summary tokens (data_p6/tok/<split>_{doc,sum}.npz)."""
64-    def __init__(self, doc_npz, sum_npz=None):
65-        d = np.load(doc_npz); ids, off = d["ids"], d["offsets"]; n = len(off) - 1
66-        I = [ids[off[i]:off[i + 1]][:IN_MAX] for i in range(n)]
67-        self.in_ids = np.concatenate(I); self.in_off = np.concatenate([[0], np.cumsum([len(a) for a in I])])
68-        self.has_tg = sum_npz is not None
69-        if self.has_tg:
70-            s = np.load(sum_npz); sids, soff = s["ids"], s["offsets"]
71-            T = [sids[soff[i]:soff[i + 1]][:TG_MAX] for i in range(n)]
72-            self.tg_ids = np.concatenate(T); self.tg_off = np.concatenate([[0], np.cumsum([len(a) for a in T])])
73-        self.doc, self.pos, self.n = np.arange(n), np.zeros((n, 2), dtype=np.int64), n

-rw-r--r--  1 root ro

In [4]:
# ---- State check before relaunch: running jobs, and which kd/m2 calibrated files exist ----
import subprocess, glob, os

# Any Phase 6 jobs or wait loops still alive? (should be none after a reset)
print(subprocess.run("ps -eo pid,etime,cmd | grep -E 'p6_run|sleep 60' | grep -v grep",
                     shell=True, capture_output=True, text=True).stdout or "no Phase 6 processes running")

# GPU use: both should be ~0 % if nothing is running
print(subprocess.run("nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)

# Which versions of the two re-decode runs exist: original (-1.25), _b125 backup, or both
W = "/kaggle/working"
for n in ("kd_m1x6__cal", "m2_houlsby__cal"):
    hits = (glob.glob(f"{W}/stats/p6_runs/{n}*") + glob.glob(f"{W}/p6_gen/{n}*")
            + glob.glob(f"{W}/submissions_p6/p6_{n}*"))
    print(n, "->", [h.replace(W + "/", "") for h in hits] or "NOTHING FOUND")

# The afternoon logs, so we can see how far each job got before the reset
print(subprocess.run(f"tail -n 3 {W}/stats/p6redo_gpu*.log {W}/stats/p6new_gpu*.log",
                     shell=True, capture_output=True, text=True).stdout)

no Phase 6 processes running
0, 0 %, 0 MiB
1, 0 %, 0 MiB

kd_m1x6__cal -> ['stats/p6_runs/kd_m1x6__cal_b125.json', 'stats/p6_runs/kd_m1x6__cal.json', 'p6_gen/kd_m1x6__cal_b125.json', 'p6_gen/kd_m1x6__cal.json', 'submissions_p6/p6_kd_m1x6__cal', 'submissions_p6/p6_kd_m1x6__cal_b125']
m2_houlsby__cal -> ['stats/p6_runs/m2_houlsby__cal.json', 'stats/p6_runs/m2_houlsby__cal_b125.json', 'p6_gen/m2_houlsby__cal.json', 'p6_gen/m2_houlsby__cal_b125.json', 'submissions_p6/p6_m2_houlsby__cal_b125', 'submissions_p6/p6_m2_houlsby__cal']
==> /kaggle/working/stats/p6redo_gpu0.log <==
10:14:06    -> eos_bias -1.0
10:24:34 DONE kd_m1x6__cal: val R1 0.5421 R2 0.2923 RL 0.2843 | length x1.10 | 417.8 tok/s | 11.5 min
10:24:34 PLAN DONE

==> /kaggle/working/stats/p6redo_gpu1.log <==
10:16:19    -> eos_bias -1.0
10:53:42 DONE m2_houlsby__cal: val R1 0.5067 R2 0.2676 RL 0.2615 | length x0.99 | 98.3 tok/s | 40.6 min
10:53:42 PLAN DONE

==> /kaggle/working/stats/p6new_gpu0.log <==
10:59:24    step 1750/6107 |

In [9]:
# ---- Queue m3_trunc on GPU 0: wait for kd_m1x6 re-decode, then train, then calibrate + decode test ----
import json, subprocess

WORK = "/kaggle/working"
train_plan = f"{WORK}/stats/p6_trunc.json"

# Show the training job so we know what will run
plan = json.load(open(train_plan))
print(json.dumps(plan, indent=2))

# Calibration job in the same format as the other runs (skipped if the plan already includes one)
cal_plan = f"{WORK}/stats/p6cal_trunc.json"
if not any(j.get("type") == "cal" for j in plan):
    json.dump([{"type": "cal", "name": "m3_trunc", "biases": [-0.75, -1.0, -1.25, -1.5], "n_docs": 20}],
              open(cal_plan, "w"), indent=2)
    steps = f"python3 p6_run.py --plan {train_plan} && python3 p6_run.py --plan {cal_plan}"
else:
    steps = f"python3 p6_run.py --plan {train_plan}"

# Background chain: poll every 60 s until GPU 0's current job finishes, then run the steps on GPU 0
chain = (f"while ! grep -q 'PLAN DONE' {WORK}/stats/p6redo_gpu0.log; do sleep 60; done; "
         f"cd {WORK}/src && export CUDA_VISIBLE_DEVICES=0 && {steps}")
subprocess.run(f"nohup bash -c \"{chain}\" > {WORK}/stats/p6_trunc_gpu0.log 2>&1 &", shell=True)
print("queued m3_trunc on GPU 0 -> log: stats/p6_trunc_gpu0.log")

[
  {
    "name": "m3_trunc",
    "ckpt": "m3",
    "method": "full",
    "lr": 0.0001,
    "mode": "trunc",
    "epochs": 2
  },
  {
    "type": "cal",
    "name": "m3_trunc",
    "biases": [
      -0.5,
      -0.75,
      -1.0,
      -1.25
    ],
    "n_docs": 20
  }
]
queued m3_trunc on GPU 0 -> log: stats/p6_trunc_gpu0.log


In [10]:
!tail -n 3 /kaggle/working/stats/p6redo_gpu*.log /kaggle/working/stats/p6_trunc_gpu0.log

==> /kaggle/working/stats/p6redo_gpu0.log <==
10:13:06 === kd_m1x6__cal: eos_bias sweep [-1.0] on 20 val documents
10:14:06    eos_bias -1.0: R1 0.5389 RL 0.2738 | length x0.95
10:14:06    -> eos_bias -1.0

==> /kaggle/working/stats/p6redo_gpu1.log <==
10:13:08 === m2_houlsby__cal: eos_bias sweep [-1.0] on 20 val documents
10:16:19    eos_bias -1.0: R1 0.5230 RL 0.2800 | length x0.89
10:16:19    -> eos_bias -1.0

==> /kaggle/working/stats/p6_trunc_gpu0.log <==


In [11]:
# ---- Show the Phase 6 training plan format (used to write the plans for the 5 new runs) ----
import json; print(json.dumps(json.load(open("/kaggle/working/stats/p6_gpu0.json")), indent=2))

[
  {
    "name": "m1_full",
    "ckpt": "m1",
    "method": "full",
    "lr": 0.0001
  },
  {
    "name": "kd_m1x6",
    "ckpt": "m1",
    "method": "full",
    "lr": 0.0002,
    "teacher": "m1_full",
    "student_layers": [
      0,
      2,
      4,
      6,
      8,
      10
    ]
  },
  {
    "name": "m1_qlora",
    "ckpt": "m1",
    "method": "qlora",
    "lr": 0.0005
  },
  {
    "name": "m3_trunc",
    "ckpt": "m3",
    "method": "full",
    "lr": 0.0001,
    "mode": "trunc",
    "epochs": 2
  }
]


In [5]:
# ---- Write Phase 6 plans for the 5 new runs (train + calibrate); nothing is launched here ----
import json, re, subprocess

WORK = "/kaggle/working"
S = f"{WORK}/stats"

# Reuse the learning rates of the existing PEFT runs so the comparison is fair
old = {j["name"]: j for j in json.load(open(f"{S}/p6_gpu1.json"))}
print("existing GPU-1 runs:", {n: (j["method"], j["lr"]) for n, j in old.items()})
lr_lora    = old.get("m4_lora", {}).get("lr", 5e-4)      # same as m4_lora
lr_houlsby = old.get("m2_houlsby", {}).get("lr", 5e-4)   # same as m2_houlsby

# Methods the Phase 6 code actually supports (quoted strings in the runner / PEFT code)
src = subprocess.run(f"cat {WORK}/src/p6_run.py {WORK}/src/p6_sum.py {WORK}/src/peft*.py 2>/dev/null",
                     shell=True, capture_output=True, text=True).stdout
known = set(re.findall(r"['\"](full|lora|lorma|qlora|ia3|houlsby|pfeiffer|adapter\w*)['\"]", src))
print("methods found in code:", sorted(known))

# The 5 new runs, split so each GPU finishes at about the same time (m2_full is the slowest: MoE)
NEW = {
    0: [{"name": "m2_full",    "ckpt": "m2", "method": "full",    "lr": 1e-4},
        {"name": "m1_ia3",     "ckpt": "m1", "method": "ia3",     "lr": 3e-3}],
    1: [{"name": "m4_full",    "ckpt": "m4", "method": "full",    "lr": 1e-4},
        {"name": "m1_lora",    "ckpt": "m1", "method": "lora",    "lr": lr_lora},
        {"name": "m1_houlsby", "ckpt": "m1", "method": "houlsby", "lr": lr_houlsby}],
}

for gpu, jobs in NEW.items():
    # Flag any method the code doesn't recognise before it wastes a GPU slot
    for j in jobs:
        if j["method"] not in known:
            print(f"  !! {j['name']}: method '{j['method']}' not found in code - check its name before launch")
    # Training plan, then a calibration plan with the wider grid (down to -1.5)
    json.dump(jobs, open(f"{S}/p6new_gpu{gpu}.json", "w"), indent=2)
    json.dump([{"type": "cal", "name": j["name"], "biases": [-0.75, -1.0, -1.25, -1.5], "n_docs": 20}
               for j in jobs], open(f"{S}/p6newcal_gpu{gpu}.json", "w"), indent=2)
    print(f"GPU {gpu}: {[j['name'] for j in jobs]} -> p6new_gpu{gpu}.json + p6newcal_gpu{gpu}.json")

existing GPU-1 runs: {'m1_lorma': ('lorma_plus', 0.0005), 'm4_lora': ('lora', 0.0005), 'm3_full': ('full', 0.0001), 'm2_houlsby': ('houlsby', 0.0005), 'm3_pfeiffer': ('pfeiffer', 0.0005)}
methods found in code: ['full', 'houlsby', 'lora', 'pfeiffer', 'qlora']
  !! m1_ia3: method 'ia3' not found in code - check its name before launch
GPU 0: ['m2_full', 'm1_ia3'] -> p6new_gpu0.json + p6newcal_gpu0.json
GPU 1: ['m4_full', 'm1_lora', 'm1_houlsby'] -> p6new_gpu1.json + p6newcal_gpu1.json


In [6]:
# ---- Swap the better -1.25 outputs back in, then launch the remaining Phase 6 jobs with HF backups ----
import os, json, glob, shutil, subprocess
from kaggle_secrets import UserSecretsClient

WORK, S = "/kaggle/working", "/kaggle/working/stats"
REPO, REPO_TYPE = "Utkarsh-1718-1/cs781-phase1-work", "model"   # set REPO_TYPE to match your restore cell

# 1) Swap: current __cal (the -1.0 re-decode) -> __cal_b100, and __cal_b125 (better val R1) -> __cal
for n in ("kd_m1x6", "m2_houlsby"):
    r = f"{n}__cal"
    for cur, b100, b125 in [(f"{S}/p6_runs/{r}.json",        f"{S}/p6_runs/{r}_b100.json",        f"{S}/p6_runs/{r}_b125.json"),
                            (f"{WORK}/p6_gen/{r}.json",      f"{WORK}/p6_gen/{r}_b100.json",      f"{WORK}/p6_gen/{r}_b125.json"),
                            (f"{WORK}/submissions_p6/p6_{r}", f"{WORK}/submissions_p6/p6_{r}_b100", f"{WORK}/submissions_p6/p6_{r}_b125")]:
        if os.path.exists(cur) and os.path.exists(b125) and not os.path.exists(b100):
            shutil.move(cur, b100); shutil.move(b125, cur)
            print("swapped:", os.path.basename(cur), "<- -1.25 version")

# 2) Pre-flight: plans from Cell B and the Trunc fix must be in place
need = [f"{WORK}/data_p6/tok/train_doc.npz", f"{S}/p6_trunc.json", f"{S}/p6new_gpu0.json", f"{S}/p6new_gpu1.json"]
missing = [p for p in need if not os.path.exists(p)]
assert not missing, f"missing: {missing} - run Cell B first"
assert 'ids, off = d["ids"]' in open(f"{WORK}/src/p6_sum.py").read(), "Trunc fix missing"

# 3) HF backup script (token reaches the background jobs through the environment only; never printed)
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
open(f"{WORK}/hf_backup.py", "w").write(f'''
import sys, os, glob, time
from huggingface_hub import HfApi
api, W = HfApi(token=os.environ["HF_TOKEN"]), "{WORK}"
paths = [f"{{W}}/{{d}}" for d in ("src", "stats", "p6_gen", "submissions_p6")]
paths += [p for n in sys.argv[1:] for p in glob.glob(f"{{W}}/p6_ckpt/{{n}}*")]
for p in paths:
    rel = os.path.relpath(p, W)
    for attempt in range(3):
        try:
            kw = dict(path_in_repo=rel, repo_id="{REPO}", repo_type="{REPO_TYPE}", commit_message="auto backup " + rel)
            api.upload_folder(folder_path=p, **kw) if os.path.isdir(p) else api.upload_file(path_or_fileobj=p, **kw)
            break
        except Exception as e:
            print("retry", rel, type(e).__name__); time.sleep(30)
print("backup ok", sys.argv[1:])
''')

# 4) m3_trunc calibration plan (wider grid, down to -1.5)
json.dump([{"type": "cal", "name": "m3_trunc", "biases": [-0.75, -1.0, -1.25, -1.5], "n_docs": 20}],
          open(f"{S}/p6cal_trunc.json", "w"))

run = lambda plan: f"python3 p6_run.py --plan {S}/{plan}"
bk  = lambda *ck: f"python3 {WORK}/hf_backup.py {' '.join(ck)}"

# 5) One chain per GPU (no re-decodes this time); first backup saves the swap and Cell B plans right away
STEPS = {
    0: [bk(), run("p6_trunc.json"), bk("m3_trunc"), run("p6cal_trunc.json"), bk(),
        run("p6new_gpu0.json"), bk("m2_full", "m1_ia3"), run("p6newcal_gpu0.json"), bk()],
    1: [run("p6new_gpu1.json"), bk("m4_full", "m1_lora", "m1_houlsby"), run("p6newcal_gpu1.json"), bk()],
}
for gpu, steps in STEPS.items():
    chain = f"cd {WORK}/src && export CUDA_VISIBLE_DEVICES={gpu}; " + "; ".join(steps)
    subprocess.run(f'nohup bash -c "{chain}" > {S}/p6all_gpu{gpu}.log 2>&1 &', shell=True)
    print(f"launched GPU {gpu} -> stats/p6all_gpu{gpu}.log")

swapped: kd_m1x6__cal.json <- -1.25 version
swapped: kd_m1x6__cal.json <- -1.25 version
swapped: p6_kd_m1x6__cal <- -1.25 version
swapped: m2_houlsby__cal.json <- -1.25 version
swapped: m2_houlsby__cal.json <- -1.25 version
swapped: p6_m2_houlsby__cal <- -1.25 version
launched GPU 0 -> stats/p6all_gpu0.log
launched GPU 1 -> stats/p6all_gpu1.log


In [8]:
!tail -n 3 /kaggle/working/stats/p6all_gpu*.log

==> /kaggle/working/stats/p6all_gpu0.log <==
backup ok []
11:33:38 === m2_full: m2 | full | chunk | lr 0.0001
11:33:40 trainable 270.396M of 270.4M | 6107 steps

==> /kaggle/working/stats/p6all_gpu1.log <==
11:33:05 === m4_full: m4 | full | chunk | lr 0.0001
11:33:08 trainable 100.292M of 100.3M | 6107 steps
11:34:55    step 250/6107 | loss 1.5832 | 107s


In [7]:
# ---- Verify the calibrated test predictions and build upload zips in priority order ----
import json, zipfile, os, numpy as np

WORK = "/kaggle/working"
# Test IDs in the order the runner uses (same source as IDS["test"] in p6_run.py)
test_ids = [m["id"] for m in json.load(open(f"{WORK}/data_p6/clean/test.json"))]

# Upload order: best validation ROUGE-1 first
ORDER = ["m1_full", "kd_m1x6", "m4_lora", "m2_houlsby", "m1_lorma", "m1_qlora", "m3_full", "m3_pfeiffer"]

os.makedirs(f"{WORK}/upload_p6", exist_ok=True)
print(f"{'#':>2} {'run':<20} {'n':>4} {'ok':>3} {'test words':>10} {'val words':>9}  zip")
for k, n in enumerate(ORDER, 1):
    run = f"{n}__cal"
    pred = json.load(open(f"{WORK}/submissions_p6/p6_{run}/prediction.json"))

    # Checks: same ID set as the test file, no duplicates, no empty summaries
    ids = [p["id"] for p in pred]
    ok = (sorted(ids) == sorted(test_ids) and len(ids) == len(set(ids))
          and all(p["summary"].strip() for p in pred))

    # Median summary length on test vs val (from p6_gen) — a large gap means test decoding drifted
    gen = json.load(open(f"{WORK}/p6_gen/{run}.json"))
    tw = int(np.median([len(p["summary"].split()) for p in pred]))
    vw = int(np.median([len(s.split()) for s in gen["val"].values()]))

    # Zip with prediction.json at the root of the archive, numbered by upload order
    zp = f"{WORK}/upload_p6/{k}_p6_{run}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{WORK}/submissions_p6/p6_{run}/prediction.json", "prediction.json")

    print(f"{k:>2} {run:<20} {len(pred):>4} {'Y' if ok else 'NO':>3} {tw:>10} {vw:>9}  {os.path.basename(zp)}")

 # run                     n  ok test words val words  zip
 1 m1_full__cal          100   Y        680       742  1_p6_m1_full__cal.zip
 2 kd_m1x6__cal          100   Y        838       838  2_p6_kd_m1x6__cal.zip
 3 m4_lora__cal          100   Y        704       783  3_p6_m4_lora__cal.zip
 4 m2_houlsby__cal       100   Y        874       781  4_p6_m2_houlsby__cal.zip
 5 m1_lorma__cal         100   Y        609       624  5_p6_m1_lorma__cal.zip
 6 m1_qlora__cal         100   Y        639       659  6_p6_m1_qlora__cal.zip
 7 m3_full__cal          100   Y        922       923  7_p6_m3_full__cal.zip
 8 m3_pfeiffer__cal      100   Y        548       670  8_p6_m3_pfeiffer__cal.zip


In [13]:
# ---- Queue the 5 new Phase 6 runs: each GPU starts when its current jobs have fully finished ----
import subprocess

WORK = "/kaggle/working"
S = f"{WORK}/stats"

# What each GPU waits for. The [x] bracket keeps the wait loop from matching its own command line.
WAIT = {0: "[p]6_trunc.json|[p]6cal_trunc.json",   # m3_trunc train + calibration chain
        1: "[p]6redo_gpu1.json"}                   # m2_houlsby re-decode

for gpu, pattern in WAIT.items():
    chain = (f"while pgrep -f '{pattern}' > /dev/null; do sleep 60; done; "     # wait for the GPU to free up
             f"cd {WORK}/src && export CUDA_VISIBLE_DEVICES={gpu}; "
             f"python3 p6_run.py --plan {S}/p6new_gpu{gpu}.json; "              # train the new runs
             f"python3 p6_run.py --plan {S}/p6newcal_gpu{gpu}.json")            # calibrate + decode test
    subprocess.run(f'nohup bash -c "{chain}" > {S}/p6new_gpu{gpu}.log 2>&1 &', shell=True)
    print(f"queued on GPU {gpu} -> log: stats/p6new_gpu{gpu}.log")

# Sanity check: the wait loops are running and see the current jobs
print(subprocess.run("ps -eo pid,etime,cmd | grep -E 'p6_run|sleep 60' | grep -v grep",
                     shell=True, capture_output=True, text=True).stdout)

queued on GPU 0 -> log: stats/p6new_gpu0.log
queued on GPU 1 -> log: stats/p6new_gpu1.log
    139       12:21 python3 p6_run.py --plan /kaggle/working/stats/p6redo_gpu1.json
    218       00:52 sleep 60



In [1]:
!ls -la --time-style=+%H:%M /kaggle/working/stats/p6all_gpu*.log 2>/dev/null && tail -n 4 /kaggle/working/stats/p6all_gpu*.log || echo "working dir was wiped -> run your restore cell"

-rw-r--r-- 1 root root 58135 12:56 /kaggle/working/stats/p6all_gpu0.log
-rw-r--r-- 1 root root  2142 12:56 /kaggle/working/stats/p6all_gpu1.log
==> /kaggle/working/stats/p6all_gpu0.log <==
12:34:05    step 5500/6107 | loss 1.3365 | 3625s
12:36:56    step 5750/6107 | loss 1.3692 | 3796s
12:39:43    step 6000/6107 | loss 1.3401 | 3964s
12:41:42    val target NLL 1.2416 | training 68.1 min

==> /kaggle/working/stats/p6all_gpu1.log <==
12:48:25    step 2500/6107 | loss 1.4716 | 871s
12:49:53    step 2750/6107 | loss 1.4255 | 960s
12:51:21    step 3000/6107 | loss 1.4430 | 1048s
12:52:49    step 3250/6107 | loss 1.4382 | 1135s


In [2]:
# ---- Resume Phase 6: skip finished jobs, relaunch only what's left, back up to HF after every job ----
import os, json, subprocess
from kaggle_secrets import UserSecretsClient

WORK, S = "/kaggle/working", "/kaggle/working/stats"
REPO, REPO_TYPE = "Utkarsh-1718-1/cs781-phase1-work", "model"   # match your restore cell

# Make sure nothing from an earlier launch is still running (avoids duplicate jobs on a GPU)
alive = subprocess.run("pgrep -f '[p]6_run.py'", shell=True, capture_output=True, text=True).stdout.split()
assert not alive, f"Phase 6 jobs still running (PIDs {alive}) - don't relaunch, just check the logs"

# Training specs come from the existing plan files (p6_trunc.json and the Cell B plans)
spec = {j["name"]: j for f in ("p6_trunc.json", "p6new_gpu0.json", "p6new_gpu1.json")
        for j in json.load(open(f"{S}/{f}")) if j["name"] in
        ("m3_trunc", "m2_full", "m1_ia3", "m4_full", "m1_lora", "m1_houlsby")}
ORDER = {0: ["m3_trunc", "m2_full", "m1_ia3"], 1: ["m4_full", "m1_lora", "m1_houlsby"]}

# A job is finished when its generation file exists (written at the very end of each job)
trained    = lambda n: os.path.exists(f"{WORK}/p6_gen/{n}.json")
calibrated = lambda n: os.path.exists(f"{WORK}/p6_gen/{n}__cal.json")

# HF backup script (token reaches the background jobs through the environment only; never printed)
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
open(f"{WORK}/hf_backup.py", "w").write(f'''
import sys, os, glob, time
from huggingface_hub import HfApi
api, W = HfApi(token=os.environ["HF_TOKEN"]), "{WORK}"
paths = [f"{{W}}/{{d}}" for d in ("src", "stats", "p6_gen", "submissions_p6")]
paths += [p for n in sys.argv[1:] for p in glob.glob(f"{{W}}/p6_ckpt/{{n}}*")]
for p in paths:
    rel = os.path.relpath(p, W)
    for attempt in range(3):
        try:
            kw = dict(path_in_repo=rel, repo_id="{REPO}", repo_type="{REPO_TYPE}", commit_message="auto backup " + rel)
            api.upload_folder(folder_path=p, **kw) if os.path.isdir(p) else api.upload_file(path_or_fileobj=p, **kw)
            break
        except Exception as e:
            print("retry", rel, type(e).__name__); time.sleep(30)
print("backup ok", sys.argv[1:])
''')

# Build each GPU's chain: per model, train (if needed) then calibrate (if needed), backing up after each
print(f"{'model':<12} {'trained':>8} {'calibrated':>11}")
for gpu, names in ORDER.items():
    steps = []
    for n in names:
        print(f"{n:<12} {str(trained(n)):>8} {str(calibrated(n)):>11}   (GPU {gpu})")
        if not trained(n):
            json.dump([spec[n]], open(f"{S}/p6res_{n}.json", "w"))
            steps += [f"python3 p6_run.py --plan {S}/p6res_{n}.json", f"python3 {WORK}/hf_backup.py {n}"]
        if not calibrated(n):
            json.dump([{"type": "cal", "name": n, "biases": [-0.75, -1.0, -1.25, -1.5], "n_docs": 20}],
                      open(f"{S}/p6rescal_{n}.json", "w"))
            steps += [f"python3 p6_run.py --plan {S}/p6rescal_{n}.json", f"python3 {WORK}/hf_backup.py"]
    if steps:
        chain = f"cd {WORK}/src && export CUDA_VISIBLE_DEVICES={gpu}; " + "; ".join(steps)
        subprocess.run(f'nohup bash -c "{chain}" >> {S}/p6res_gpu{gpu}.log 2>&1 &', shell=True)
        print(f"-> GPU {gpu}: launched {len(steps)//2} steps, log stats/p6res_gpu{gpu}.log")
    else:
        print(f"-> GPU {gpu}: everything already done")

# Record free RAM every 5 min, so if the session dies again we can see whether memory ran out
subprocess.run(f"nohup bash -c 'while true; do date +%H:%M; free -g | sed -n 2p; sleep 300; done' "
               f">> {S}/mem.log 2>&1 &", shell=True)

model         trained  calibrated
m3_trunc         True        True   (GPU 0)
m2_full         False       False   (GPU 0)
m1_ia3          False       False   (GPU 0)
-> GPU 0: launched 4 steps, log stats/p6res_gpu0.log
m4_full          True       False   (GPU 1)
m1_lora         False       False   (GPU 1)
m1_houlsby      False       False   (GPU 1)
-> GPU 1: launched 5 steps, log stats/p6res_gpu1.log


CompletedProcess(args="nohup bash -c 'while true; do date +%H:%M; free -g | sed -n 2p; sleep 300; done' >> /kaggle/working/stats/mem.log 2>&1 &", returncode=0)

In [3]:
# ---- m2_full: if its trained checkpoint survived, skip retraining and go straight to calibration ----
import glob, os, subprocess, datetime

WORK, S = "/kaggle/working", "/kaggle/working/stats"

# New results so far
print(subprocess.run(f"grep -h 'DONE' {S}/p6all_gpu*.log {S}/p6res_gpu*.log", shell=True,
                     capture_output=True, text=True).stdout)

# m2_full checkpoint and when it was written (container clock = UTC; training ended 12:41 UTC)
ck = sorted(glob.glob(f"{WORK}/p6_ckpt/m2_full*"))
for c in ck:
    t = datetime.datetime.fromtimestamp(os.path.getmtime(c))
    print(f"{os.path.basename(c)}  written {t:%H:%M} UTC  {os.path.getsize(c)/1e6:.0f} MB")
fresh = bool(ck) and all(datetime.datetime.fromtimestamp(os.path.getmtime(c)).strftime("%H:%M") >= "12:41" for c in ck)

if fresh:
    # Stop GPU 0's chain (its retraining of m2_full) - the chain and its python both mention p6res_m2_full
    subprocess.run("pkill -f '[p]6res_m2_full'", shell=True)
    # New GPU 0 chain: calibrate m2_full from the existing checkpoint, then train + calibrate m1_ia3
    steps = [f"python3 p6_run.py --plan {S}/p6rescal_m2_full.json", f"python3 {WORK}/hf_backup.py m2_full",
             f"python3 p6_run.py --plan {S}/p6res_m1_ia3.json",     f"python3 {WORK}/hf_backup.py m1_ia3",
             f"python3 p6_run.py --plan {S}/p6rescal_m1_ia3.json",  f"python3 {WORK}/hf_backup.py"]
    chain = f"cd {WORK}/src && export CUDA_VISIBLE_DEVICES=0; " + "; ".join(steps)
    subprocess.run(f'nohup bash -c "{chain}" >> {S}/p6res_gpu0.log 2>&1 &', shell=True)
    print("GPU 0: skipped m2_full retraining -> calibrating from the saved checkpoint")
else:
    print("no complete m2_full checkpoint -> leave the retraining running")

11:33:09 PLAN DONE
11:33:26 PLAN DONE
12:33:51 DONE m4_full: val R1 0.4700 R2 0.2579 RL 0.2679 | length x0.66 | 161.8 tok/s | 60.8 min

m2_full.pt  written 12:56 UTC  541 MB
GPU 0: skipped m2_full retraining -> calibrating from the saved checkpoint


In [4]:
# ---- Why did m3_trunc fail in seconds at 11:33? ----
import subprocess, os, datetime
S, W = "/kaggle/working/stats", "/kaggle/working"

# The error: everything m3_trunc printed at the start of the GPU-0 log
print(subprocess.run(f"grep -n -m 1 -B 2 -A 30 -E 'Traceback|Error|m3_trunc' {S}/p6all_gpu0.log",
                     shell=True, capture_output=True, text=True).stdout)

# Where the m3_trunc files the resume cell found came from (timestamps in UTC)
for f in [f"{W}/p6_gen/m3_trunc.json", f"{W}/p6_gen/m3_trunc__cal.json", f"{S}/p6_runs/m3_trunc.json"]:
    if os.path.exists(f):
        t = datetime.datetime.fromtimestamp(os.path.getmtime(f))
        print(f"{os.path.relpath(f, W)}  {t:%d %b %H:%M}  {os.path.getsize(f)/1024:.0f} KB")

739-
740-
741:
  ..._m3_trunc/p6_m3_trunc.zip: 100%|██████████| 34.3kB / 34.3kB            
742-
743-
744-
745-
746-
747-
748-
749-
750-
751-
752-
  ..._cal/p6_m3_trunc__cal.zip: 100%|██████████| 39.1kB / 39.1kB            
753-
754-
  ..._m1_qlora/p6_m1_qlora.zip: 100%|██████████| 34.5kB / 34.5kB            
755-
756-
757-
  ...__cal/p6_m3_full__cal.zip: 100%|██████████|  187kB /  187kB            
758-
759-
760-
761-
  ...houlsby/p6_m2_houlsby.zip: 100%|██████████| 38.5kB / 38.5kB            
762-
763-
764-
765-
766-
  ...p6_m3_full/p6_m3_full.zip: 100%|██████████| 49.0kB / 49.0kB            
767-
768-
769-
770-
771-

p6_gen/m3_trunc.json  28 Oct 12:56  286 KB
p6_gen/m3_trunc__cal.json  28 Oct 12:56  326 KB
stats/p6_runs/m3_trunc.json  28 Oct 12:56  1 KB


In [5]:
# ---- Is m3_trunc genuinely done? Show its saved scores, DONE lines, and test submission ----
import json, glob, os, subprocess
W, S = "/kaggle/working", "/kaggle/working/stats"

# Saved results for the training run and the calibrated run (R1/R2/RL, length, timings)
for f in sorted(glob.glob(f"{S}/p6_runs/m3_trunc*.json")):
    print(os.path.basename(f), "->", json.load(open(f)))

# Any DONE line for m3_trunc in any log (shows when it actually ran)
print(subprocess.run(f"grep -h 'DONE m3_trunc' {S}/*.log", shell=True,
                     capture_output=True, text=True).stdout or "no 'DONE m3_trunc' line in any log")

# Does the runner skip jobs that already have output? (explains the 17-second PLAN DONE)
print(subprocess.run(f"grep -n -i -E 'skip|exists' {W}/src/p6_run.py | head", shell=True,
                     capture_output=True, text=True).stdout)

# Calibrated test submission present?
p = f"{W}/submissions_p6/p6_m3_trunc__cal/prediction.json"
print("test submission:", f"{len(json.load(open(p)))} summaries" if os.path.exists(p) else "MISSING")

m3_trunc.json -> {'cfg': {'ckpt': 'm3', 'method': 'full', 'lr': 0.0001, 'epochs': 2, 'bs': 8, 'r': 16, 'alpha': 32, 'mode': 'trunc', 'teacher': None, 'student_layers': None, 'kd_T': 2.0, 'kd_lam': 0.5, 'seed': 0}, 'gen': {'eos_bias': 0.0, 'no_repeat': 4, 'dedupe': True}, 'trainable_M': 109.538, 'total_M': 109.5, 'model_MB': 438.2, 'val_rouge': {'rouge1': 0.3437405200618535, 'rouge2': 0.12934461274719947, 'rougeL': 0.19680683966064105, 'len_ratio': 0.46111875580583495}, 'gen_tok_per_s': 747.5, 'test_gen_min': 0.6, 'test_words_median': 264, 'min': 12.3, 'hist': [{'step': 250, 'loss': 2.5317369050979615}, {'step': 500, 'loss': 2.4911500577926637}, {'step': 750, 'loss': 2.4643182306289675}, {'step': 1000, 'loss': 2.1967390999794008}, {'step': 1250, 'loss': 2.1404864797592165}, {'step': 1500, 'loss': 2.126305139541626}], 'val_nll': 2.449620609283447, 'train_min': 10.9}
m3_trunc__cal.json -> {'cfg': {'ckpt': 'm3', 'method': 'full', 'lr': 0.0001, 'epochs': 2, 'bs': 8, 'r': 16, 'alpha': 32, 'm

In [6]:
# ---- Zip m3_trunc__cal for upload (baseline entry) ----
import zipfile
W = "/kaggle/working"
with zipfile.ZipFile(f"{W}/upload_p6/9_p6_m3_trunc__cal.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(f"{W}/submissions_p6/p6_m3_trunc__cal/prediction.json", "prediction.json")
print("ready: upload_p6/9_p6_m3_trunc__cal.zip")

ready: upload_p6/9_p6_m3_trunc__cal.zip


In [8]:
# ---- Unblock m2_full calibration: write the base-run record the cal job waits for ----
import json, copy, os, time, subprocess
S = "/kaggle/working/stats/p6_runs"

assert not os.path.exists(f"{S}/m2_full.json"), "record already exists - nothing to do"

# m2_full used exactly m1_full's training config except the base checkpoint (m2 instead of m1)
rec = copy.deepcopy(json.load(open(f"{S}/m1_full.json")))
rec["cfg"]["ckpt"] = "m2"
# Drop m1_full's own scores so nothing reports them as m2_full's; mark the record as reconstructed
for k in ("val_rouge", "hist", "val_nll", "gen_tok_per_s", "test_gen_min", "test_words_median", "min", "train_min"):
    rec.pop(k, None)
rec["note"] = "reconstructed: training finished 12:41 UTC (val NLL 1.2416, 68.1 min); kernel died during the uncalibrated decode"
json.dump(rec, open(f"{S}/m2_full.json", "w"), indent=2)
print("wrote", f"{S}/m2_full.json", "| cfg:", rec["cfg"])

# Give the waiting cal job a moment to notice, then show whether it started
time.sleep(90)
print(subprocess.run("tail -n 4 /kaggle/working/stats/p6res_gpu0.log; "
                     "nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)

wrote /kaggle/working/stats/p6_runs/m2_full.json | cfg: {'ckpt': 'm2', 'method': 'full', 'lr': 0.0001, 'epochs': 1, 'bs': 8, 'r': 16, 'alpha': 32, 'mode': 'chunk', 'teacher': None, 'student_layers': None, 'kd_T': 2.0, 'kd_lam': 0.5, 'seed': 0}
12:58:48 === m2_full: m2 | full | chunk | lr 0.0001
12:58:52 trainable 270.396M of 270.4M | 6107 steps
0, 0 %, 0 MiB
1, 48 %, 1379 MiB



In [42]:
# ---- Phase 6 monitor: what each GPU is running, progress, memory, and finished results ----
import subprocess, glob, json, os, time, datetime

W, S = "/kaggle/working", "/kaggle/working/stats"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
print("now:", datetime.datetime.utcnow().strftime("%H:%M UTC"), "| IST = UTC + 5:30\n")

# 1) GPU load and memory
print("GPU  util  memory")
print(sh("nvidia-smi --query-gpu=index,utilization.gpu,memory.used,memory.total --format=csv,noheader"), "\n")

# 2) Which job each GPU is running: read each p6_run process's plan and its CUDA_VISIBLE_DEVICES
for pid in sh("pgrep -f '[p]ython3 p6_run.py'").split():
    try:
        cmd = open(f"/proc/{pid}/cmdline").read().replace("\0", " ")
        env = dict(kv.split("=", 1) for kv in open(f"/proc/{pid}/environ").read().split("\0") if "=" in kv)
        plan = json.load(open(cmd.split("--plan")[1].split()[0]))
        jobs = ", ".join((j.get("type", "train") + ":" + j["name"]) for j in plan)
        print(f"GPU {env.get('CUDA_VISIBLE_DEVICES', '?')}  pid {pid}  running -> {jobs}")
    except Exception:
        pass
if not sh("pgrep -f '[p]ython3 p6_run.py'"):
    print("no Phase 6 job running")

# 3) Latest progress per GPU log, and how long since it last wrote (stale > 10 min = suspicious)
for f in sorted(glob.glob(f"{S}/p6res_gpu*.log")):
    age = (time.time() - os.path.getmtime(f)) / 60
    lines = [l for l in open(f, errors="ignore").read().splitlines()
             if l.strip() and "%|" not in l]                       # drop HF upload progress bars
    print(f"\n=== {os.path.basename(f)} | last write {age:.0f} min ago{'  <-- STALE?' if age > 10 else ''} ===")
    print("\n".join(lines[-3:]))

# 4) RAM (to catch the cause if the kernel dies again)
print("\nRAM (GB) total used free shared cache avail:", sh(f"tail -n 1 {S}/mem.log 2>/dev/null") or sh("free -g | sed -n 2p"))

# 5) Every finished calibrated run, best first
rows = []
for f in glob.glob(f"{S}/p6_runs/*__cal.json"):
    r = json.load(open(f)); v = r["val_rouge"]
    rows.append((v["rouge1"], os.path.basename(f)[:-5], v["rouge2"], v["rougeL"], v["len_ratio"], r["gen"]["eos_bias"]))
print(f"\n{'calibrated run':<22} {'R1':>6} {'R2':>6} {'RL':>6} {'len':>5} {'eos':>6}")
for r1, n, r2, rl, ln, eb in sorted(rows, reverse=True):
    print(f"{n:<22} {r1:6.3f} {r2:6.3f} {rl:6.3f} {ln:5.2f} {eb:6.2f}")

now: 16:27 UTC | IST = UTC + 5:30

GPU  util  memory
0, 0 %, 0 MiB, 15360 MiB
1, 0 %, 0 MiB, 15360 MiB 

no Phase 6 job running

=== p6res_gpu0.log | last write 4 min ago ===
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            
backup ok []

=== p6res_gpu1.log | last write 37 min ago  <-- STALE? ===
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            
backup ok []

RAM (GB) total used free shared cache avail: Mem:              31           0           4           0          25          30

calibrated run             R1     R2     RL   len    eos
m1_full__cal            0.577  0.323  0.305  1.19  -1.00
m4_full__cal            0.575  0.332  0.318  1.22  -0.75
m2_full__cal            0.563  0.322  0.298  1.49  -1.25
kd_m1x6__cal            0.555  0.306  0.288  1.36  -1.25
m4_lora__cal            0.552  0.2

/tmp/ipykernel_57/2767944789.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print("now:", datetime.datetime.utcnow().strftime("%H:%M UTC"), "| IST = UTC + 5:30\n")


In [27]:
# ---- Find the m1_ia3 startup error and how IA3 is named in the code ----
import subprocess
S, SRC = "/kaggle/working/stats", "/kaggle/working/src"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout

# Last error in the GPU-0 log (progress bars removed), with the lines around it
log = [l for l in open(f"{S}/p6res_gpu0.log", errors="ignore").read().splitlines() if "%|" not in l and l.strip()]
hits = [i for i, l in enumerate(log) if any(k in l for k in ("Traceback", "Error", "error", "ia3"))]
print("\n".join(log[max(0, hits[-1] - 25): hits[-1] + 5]) if hits else "no error text found in the log")

# How the code spells and dispatches IA3 (case-insensitive), so the plan can use the exact name
print("\n--- IA3 in source ---")
print(sh(f"grep -n -i 'ia3\\|ia_3\\|IA3' {SRC}/*.py | head -20") or "no IA3 code found in src/")

# Which method names the Phase 6 runner accepts
print("--- method dispatch in p6_run.py / p6_sum.py ---")
print(sh(f"grep -n -E 'method.*(==|in) ' {SRC}/p6_run.py {SRC}/p6_sum.py | head -20"))

backup ok ['m2_full']
13:52:29 !!! m1_ia3 FAILED
Traceback (most recent call last):
  File "/kaggle/working/src/p6_run.py", line 195, in <module>
    run(name, **job)
  File "/kaggle/working/src/p6_run.py", line 117, in run
    m = build(ckpt, method, student_layers, r, alpha)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/src/p6_run.py", line 72, in build
    return S.apply_method(m, method, r=r, alpha=alpha)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/src/p6_sum.py", line 245, in apply_method
    raise ValueError(method)
ValueError: ia3
13:52:29 PLAN DONE
No files have been modified since last commit. Skipping to prevent empty commit.
It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, che

In [28]:
# ---- Show p6_sum.apply_method and the Phase 3 IA3 code, to wire IA3 into Phase 6 ----
import subprocess
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout

# Phase 6 dispatcher (where the new ia3 branch goes)
print("=== p6_sum.py 205-250 ===")
print(sh("sed -n '205,250p' /kaggle/working/src/p6_sum.py"))

# Phase 3 IA3 modules and how they're attached (the code to reuse)
print("=== p3_rr.py 150-200 ===")
print(sh("sed -n '150,200p' /kaggle/working/src/p3_rr.py"))

=== p6_sum.py 205-250 ===
def quantize_nf4(m):
    n = 0
    for L in layers(m):
        for mod in list(L.modules()):
            for name, child in list(mod.named_children()):
                if isinstance(child, nn.Linear) and name != "router":
                    setattr(mod, name, NF4Linear(child)); n += 1
    return n


def apply_method(m, method, r=16, alpha=32, bottleneck=64, n_prompt=20):
    """Freeze the Phase-1 model and add the method's trainable parameters."""
    if method == "full":
        for p in m.parameters():
            p.requires_grad = True
        return m
    for p in m.parameters():
        p.requires_grad = False
    if method == "lora":
        _wrap(m, lambda l: pp.LoRALinear(l, r, alpha), LORA_TARGETS)
    elif method == "lorma_plus":
        _wrap(m, lambda l: pp.LoRMAPlusLinear(l, r, alpha), LORA_TARGETS)
    elif method == "lorma_pi":
        _wrap(m, lambda l: pp.LoRMAPiLinear(l, r), LORA_TARGETS)
    elif method in ("houlsby", "pfeiffer"):
        a

In [29]:
# ---- Wire IA3 into Phase 6 (p6_sum.apply_method), then relaunch m1_ia3 on GPU 0 ----
import subprocess, time

W, S = "/kaggle/working", "/kaggle/working/stats"
P = f"{W}/src/p6_sum.py"
src = open(P).read()
assert 'method == "ia3"' not in src, "IA3 branch already present"

# 1) The two IA3 modules (same as p3_rr.py), placed just before apply_method
ia3_classes = '''class IA3Out(nn.Module):
    """IA^3: y = (W x) * l  (rescales keys / values)."""
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l


class IA3In(nn.Module):
    """IA^3: y = W (x * l)  (rescales the FFN intermediate activation before the down-projection)."""
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l)


def apply_method(m, method'''

# 2) The dispatch branch, placed just before the final "else: raise ValueError(method)"
ia3_branch = '''    elif method == "ia3":                                             # IA^3 (Liu et al., 2022)
        for L in layers(m):
            for mod in list(L.modules()):
                for name in ("wk", "wv", "w_uk", "w_uv"):              # keys / values (dense + MLA)
                    lin = getattr(mod, name, None)
                    if isinstance(lin, nn.Linear):
                        setattr(mod, name, IA3Out(lin))
                if type(mod).__name__ == "SwiGLU" and isinstance(getattr(mod, "w2", None), nn.Linear):
                    mod.w2 = IA3In(mod.w2)                             # FFN intermediate activation
    else:
        raise ValueError(method)'''

assert src.count("def apply_method(m, method") == 1 and src.count("    else:\n        raise ValueError(method)") == 1
src = src.replace("def apply_method(m, method", ia3_classes, 1)
src = src.replace("    else:\n        raise ValueError(method)", ia3_branch, 1)
open(P, "w").write(src)
print("IA3 wired into p6_sum.apply_method")

# 3) Relaunch on GPU 0: train -> backup -> calibrate -> backup
steps = [f"python3 p6_run.py --plan {S}/p6res_m1_ia3.json",    f"python3 {W}/hf_backup.py m1_ia3",
         f"python3 p6_run.py --plan {S}/p6rescal_m1_ia3.json", f"python3 {W}/hf_backup.py"]
chain = f"cd {W}/src && export CUDA_VISIBLE_DEVICES=0; " + "; ".join(steps)
subprocess.run(f'nohup bash -c "{chain}" >> {S}/p6res_gpu0.log 2>&1 &', shell=True)
print("launched m1_ia3 on GPU 0")

# 4) Startup check: the 'trainable' line should show a tiny fraction of the model (well under 1M)
time.sleep(120)
print(subprocess.run(f"grep -v '%|' {S}/p6res_gpu0.log | tail -n 5", shell=True,
                     capture_output=True, text=True).stdout)

IA3 wired into p6_sum.apply_method
launched m1_ia3 on GPU 0
No files have been modified since last commit. Skipping to prevent empty commit.
backup ok ['m1_ia3']
15:21:09 === m1_ia3: m1 | ia3 | chunk | lr 0.003
15:21:11 trainable 0.031M of 100.1M | 6107 steps
15:22:33    step 250/6107 | loss 2.1355 | 82s



In [43]:
# ---- Verify and zip every calibrated Phase 6 run for upload, best validation R1 first ----
import json, glob, os, zipfile

W, S = "/kaggle/working", "/kaggle/working/stats"
test_ids = sorted(m["id"] for m in json.load(open(f"{W}/data_p6/clean/test.json")))

# Every finished calibrated run, ranked by validation ROUGE-1
runs = sorted(((json.load(open(f))["val_rouge"]["rouge1"], os.path.basename(f)[:-5])
               for f in glob.glob(f"{S}/p6_runs/*__cal.json")), reverse=True)

os.makedirs(f"{W}/upload_p6", exist_ok=True)
for f in glob.glob(f"{W}/upload_p6/*.zip"): os.remove(f)          # rebuild cleanly in the new order

for k, (r1, run) in enumerate(runs, 1):
    p = f"{W}/submissions_p6/p6_{run}/prediction.json"
    pred = json.load(open(p))
    # Same checks as before: full test ID set, no duplicates, no empty summaries
    ok = sorted(x["id"] for x in pred) == test_ids and all(x["summary"].strip() for x in pred)
    zp = f"{W}/upload_p6/{k:02d}_p6_{run}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(p, "prediction.json")
    print(f"{k:2d}  {run:<20} R1 {r1:.3f}  {'OK' if ok else '!! CHECK'}  -> {os.path.basename(zp)}")

 1  m1_full__cal         R1 0.577  OK  -> 01_p6_m1_full__cal.zip
 2  m4_full__cal         R1 0.575  OK  -> 02_p6_m4_full__cal.zip
 3  m2_full__cal         R1 0.563  OK  -> 03_p6_m2_full__cal.zip
 4  kd_m1x6__cal         R1 0.555  OK  -> 04_p6_kd_m1x6__cal.zip
 5  m4_lora__cal         R1 0.552  OK  -> 05_p6_m4_lora__cal.zip
 6  m2_houlsby__cal      R1 0.534  OK  -> 06_p6_m2_houlsby__cal.zip
 7  m1_lorma__cal        R1 0.533  OK  -> 07_p6_m1_lorma__cal.zip
 8  m1_houlsby__cal      R1 0.527  OK  -> 08_p6_m1_houlsby__cal.zip
 9  m1_qlora__cal        R1 0.524  OK  -> 09_p6_m1_qlora__cal.zip
10  m1_lora__cal         R1 0.521  OK  -> 10_p6_m1_lora__cal.zip
11  m3_full__cal         R1 0.467  OK  -> 11_p6_m3_full__cal.zip
12  m3_pfeiffer__cal     R1 0.417  OK  -> 12_p6_m3_pfeiffer__cal.zip
13  m1_ia3__cal          R1 0.415  OK  -> 13_p6_m1_ia3__cal.zip
14  m3_trunc__cal        R1 0.361  OK  -> 14_p6_m3_trunc__cal.zip
